### OBJECTIVE 1 of the Master projects.

# VQA Implementation , KAK Decomposition (Pure Python)

## 1. Qubit Convention

Standard **big-endian (BE)**: first Kronecker factor $= R$, second $= A$.

$$\rho_{RA} \equiv \mathrm{kron}(R,\,A), \qquad \text{basis order: }|00\rangle,|01\rangle,|10\rangle,|11\rangle$$

No hidden index reversal anywhere.

## 2. Isometry Ansatz , Full U(4), 16 Parameters

The isometry $V : \mathbb{C}^2 \to \mathbb{C}^2 \otimes \mathbb{C}^2$ is embedded in $U \in U(4)$:

$$V|\psi\rangle_A = U\bigl(|\psi\rangle_A \otimes |0\rangle_E\bigr)$$

$U(4)$ has $\dim U(4) = 16$ real parameters via **KAK (Cartan) decomposition**:

$$\boxed{U(\boldsymbol{\theta}) = e^{i\phi}\,(A_1\otimes A_2)\cdot\exp\!\left[i(c_1\,X{\otimes}X+c_2\,Y{\otimes}Y+c_3\,Z{\otimes}Z)\right]\cdot(A_3\otimes A_4)}$$

Each $A_k \in SU(2)$ via **ZYZ Euler decomposition**:

$$A(\alpha,\beta,\gamma) = R_Z(\alpha)\,R_Y(\beta)\,R_Z(\gamma)$$

$$R_Z(\alpha)=\begin{pmatrix}e^{-i\alpha/2}&0\\0&e^{i\alpha/2}\end{pmatrix}, \qquad R_Y(\beta)=\begin{pmatrix}\cos\frac{\beta}{2}&-\sin\frac{\beta}{2}\\\sin\frac{\beta}{2}&\cos\frac{\beta}{2}\end{pmatrix}$$

Parameter vector:

$$\boldsymbol{\theta} = (\underbrace{\alpha_1,\beta_1,\gamma_1}_{A_1},\underbrace{\alpha_2,\beta_2,\gamma_2}_{A_2},\underbrace{\alpha_3,\beta_3,\gamma_3}_{A_3},\underbrace{\alpha_4,\beta_4,\gamma_4}_{A_4},\underbrace{c_1,c_2,c_3}_{\Omega},\underbrace{\phi}_{\text{phase}}) \in [-\pi,\pi]^{16}$$

Interaction term computed via `scipy.linalg.expm`:

$$\Omega(c_1,c_2,c_3) = \exp\!\left[i(c_1\,X{\otimes}X+c_2\,Y{\otimes}Y+c_3\,Z{\otimes}Z)\right]$$

Full construction:

$$U(\boldsymbol{\theta}) = e^{i\phi}\,(A_1\otimes A_2)\cdot\Omega(c_1,c_2,c_3)\cdot(A_3\otimes A_4)$$

## 3. State Evolution

$$\rho_{RAE} = \rho_{RA}\otimes|0\rangle\langle0|_E \quad \Rightarrow \quad \texttt{np.kron(rho\_RA,\ [[1,0],[0,0]])}$$

BE index assignment: subsystem $0=R$, $1=A$, $2=E$.

$$\sigma_{RBE} = (I_R\otimes U_{AE})\,\rho_{RAE}\,(I_R\otimes U_{AE})^\dagger$$



In [1]:
import numpy as np
from scipy.optimize import minimize
from scipy.linalg import expm

In [2]:
# ============================================================
# PRIMITIVES
# ============================================================

def partial_trace_out(rho, keep, dims):
    dims = list(dims)
    n    = len(dims)
    rho_t = rho.reshape(dims + dims)
    trace_out = [i for i in range(n) if i not in keep]
    for ax in sorted(trace_out, reverse=True):
        current_n = rho_t.ndim // 2
        rho_t = np.trace(rho_t, axis1=ax, axis2=ax + current_n)
    kept_dims = [dims[i] for i in keep]
    d_keep    = int(np.prod(kept_dims))
    return rho_t.reshape(d_keep, d_keep)

def von_neumann_entropy(rho):
    eigvals = np.linalg.eigvalsh(rho).real
    eigvals = eigvals[eigvals > 1e-12]
    return float(-np.sum(eigvals * np.log2(eigvals)))

def mutual_information(rho_AB, rho_A, rho_B):
    return (von_neumann_entropy(rho_A)
          + von_neumann_entropy(rho_B)
          - von_neumann_entropy(rho_AB))

In [3]:
# ============================================================
# COHERENT INFORMATION
# ============================================================

def coherent_information(rho_RA):
    """
    I_c(R>A) = S(rho_A) - S(rho_RA)
    Computed on the INPUT state before the isometry.
    Can be negative.
    """
    dims   = [2, 2]   # R=0, A=1
    rho_A  = partial_trace_out(rho_RA, keep=[1], dims=dims)
    S_A    = von_neumann_entropy(rho_A)
    S_RA   = von_neumann_entropy(rho_RA)
    return S_A - S_RA


# ============================================================
# QUANTUM DISCORD  D(R|A)
# ============================================================

def _projector(theta, phi):
    """
    Single-qubit projector |psi><psi| parametrised by Bloch angles.
    |psi> = cos(theta/2)|0> + exp(i*phi)*sin(theta/2)|1>
    """
    c = np.cos(theta / 2)
    s = np.sin(theta / 2) * np.exp(1j * phi)
    v = np.array([c, s], dtype=complex)
    return np.outer(v, v.conj())


def _classical_mutual_info(rho_RA, theta, phi):
    """
    I(R:A | {Pi_k}) for a projective measurement on A
    parametrised by (theta, phi).
    
    Layout of rho_RA: row/col order is |RA> = |R> x |A>,
    so dims = [2,2], R=0, A=1.
    
    Pi_0 = |psi><psi|,  Pi_1 = I - |psi><psi|
    
    p_k    = Tr[ (I_R x Pi_k) rho_RA ]
    rho_Rk = Tr_A[ (I_R x Pi_k) rho_RA ] / p_k
    
    Classical MI = S(rho_R) - sum_k p_k S(rho_Rk)
    """
    dims  = [2, 2]
    I2    = np.eye(2, dtype=complex)
    Pi0   = _projector(theta, phi)
    Pi1   = I2 - Pi0

    rho_R = partial_trace_out(rho_RA, keep=[0], dims=dims)
    S_R   = von_neumann_entropy(rho_R)

    conditional_entropy = 0.0
    for Pi in [Pi0, Pi1]:
        # (I_R ⊗ Pi_k) acting on rho_RA
        M         = np.kron(I2, Pi) @ rho_RA @ np.kron(I2, Pi).conj().T
        p_k       = np.real(np.trace(M))
        if p_k < 1e-12:
            continue
        rho_Rk    = partial_trace_out(M / p_k, keep=[0], dims=dims)
        conditional_entropy += p_k * von_neumann_entropy(rho_Rk)

    return S_R - conditional_entropy
    

def quantum_discord_wrt_A(rho_RA, n_grid=100, n_refine=200):
    """
    Improved discord with grid initialization over Bloch sphere.
    Grid: n_grid points in theta x n_grid points in phi.
    """
    dims  = [2, 2]
    rho_R = partial_trace_out(rho_RA, keep=[0], dims=dims)
    rho_A = partial_trace_out(rho_RA, keep=[1], dims=dims)
    I_RA  = mutual_information(rho_RA, rho_R, rho_A)

    best_cmi = -float('inf')

    def neg_cmi(x):
        return -_classical_mutual_info(rho_RA, x[0], x[1])

    # Grid initialization: evenly cover the Bloch sphere
    thetas = np.linspace(0, np.pi,    n_grid, endpoint=True)
    phis   = np.linspace(0, 2*np.pi,  n_grid, endpoint=False)

    candidates = []
    for th in thetas:
        for ph in phis:
            val = _classical_mutual_info(rho_RA, th, ph)
            candidates.append((val, th, ph))

    # Take top-5 grid points and refine each
    candidates.sort(reverse=True)
    for val, th0, ph0 in candidates[:100]:
        result = minimize(neg_cmi,
                          np.array([th0, ph0]),
                          method='L-BFGS-B',
                          bounds=[(0, np.pi), (0, 2*np.pi)],
                          options={'maxiter': 7000, 'ftol': 1e-14, 'gtol': 1e-10})
        if -result.fun > best_cmi:
            best_cmi = -result.fun

    # Additional random restarts for safety
    for _ in range(n_refine):
        x0 = np.array([np.random.uniform(0, np.pi),
                        np.random.uniform(0, 2*np.pi)])
        result = minimize(neg_cmi, x0,
                          method='L-BFGS-B',
                          bounds=[(0, np.pi), (0, 2*np.pi)],
                          options={'maxiter': 7000, 'ftol': 1e-14, 'gtol': 1e-10})
        if -result.fun > best_cmi:
            best_cmi = -result.fun

    return max(I_RA - best_cmi, 0.0)

# ============================================================
# QUANTUM DISCORD  D(A|R)  — measurement on R
# KAK notebook  (big-endian: dims=[2,2], R=0, A=1)
# ============================================================

def _classical_MI_wrt_R(rho_RA, theta, phi):
    """
    Classical mutual information J(R:A | {Pi_k^R}) for a fixed
    projective measurement on R parametrised by Bloch angles (theta, phi).

    Big-endian layout of rho_RA: kron(R, A).
    Subsystem indices: R=0, A=1.

    Pi_0 = |psi><psi|,  Pi_1 = I - |psi><psi|

    p_k    = Tr[ (Pi_k ⊗ I_A) rho_RA ]
    rho_Ak = Tr_R[ (Pi_k ⊗ I_A) rho_RA ] / p_k

    Classical MI = S(rho_A) - sum_k p_k * S(rho_Ak)
    """
    dims = [2, 2]
    I2   = np.eye(2, dtype=complex)

    c    = np.cos(theta / 2)
    s    = np.sin(theta / 2) * np.exp(1j * phi)
    v    = np.array([c, s], dtype=complex)
    Pi0  = np.outer(v, v.conj())
    Pi1  = I2 - Pi0

    # rho_A: trace out R (subsystem 0)
    rho_A = partial_trace_out(rho_RA, keep=[1], dims=dims)
    S_A   = von_neumann_entropy(rho_A)

    conditional_entropy = 0.0
    for Pi in [Pi0, Pi1]:
        # Apply Pi to R (first factor): kron(Pi, I_A)
        M   = np.kron(Pi, I2) @ rho_RA @ np.kron(Pi, I2).conj().T
        p_k = np.real(np.trace(M))
        if p_k < 1e-12:
            continue
        # Trace out R to get conditional rho_A
        rho_Ak = partial_trace_out(M / p_k, keep=[1], dims=dims)
        conditional_entropy += p_k * von_neumann_entropy(rho_Ak)

    return S_A - conditional_entropy


def quantum_discord_wrt_R(rho_RA, n_grid=100, n_refine=200):
    """
    Discord D(A|R) = I(R:A) - max_{ {Pi_k^R} } J(R:A | {Pi_k^R})

    Measurement is on R. Post-measurement states are on A.
    Grid search over Bloch sphere of R, then local refinement.

    Big-endian: kron(R, A), R=subsystem 0, A=subsystem 1.
    """
    dims  = [2, 2]
    rho_R = partial_trace_out(rho_RA, keep=[0], dims=dims)
    rho_A = partial_trace_out(rho_RA, keep=[1], dims=dims)
    I_RA  = mutual_information(rho_RA, rho_R, rho_A)

    def neg_cmi(x):
        return -_classical_MI_wrt_R(rho_RA, x[0], x[1])

    # Grid search over Bloch sphere
    thetas     = np.linspace(0, np.pi,   n_grid, endpoint=True)
    phis       = np.linspace(0, 2*np.pi, n_grid, endpoint=False)
    candidates = []
    for th in thetas:
        for ph in phis:
            val = _classical_MI_wrt_R(rho_RA, th, ph)
            candidates.append((val, th, ph))

    # Refine top-5 grid points
    candidates.sort(reverse=True)
    best_cmi = -float('inf')
    for val, th0, ph0 in candidates[:100]:
        res = minimize(neg_cmi,
                       x0=[th0, ph0],
                       method='L-BFGS-B',
                       bounds=[(0, np.pi), (0, 2*np.pi)],
                       options={'maxiter': 7000, 'ftol': 1e-14, 'gtol': 1e-10})
        if -res.fun > best_cmi:
            best_cmi = -res.fun

    discord = max(0.0, I_RA - best_cmi)
    return discord
# ============================================================
# FULL STATE DIAGNOSTICS
# ============================================================

def state_diagnostics(rho_RA, state_name):
    """
    Prints I(R:A), I_c(R>A), and D(R|A) for a given input state.
    """
    dims  = [2, 2]
    rho_R = partial_trace_out(rho_RA, keep=[0], dims=dims)
    rho_A = partial_trace_out(rho_RA, keep=[1], dims=dims)

    I_RA  = mutual_information(rho_RA, rho_R, rho_A)
    I_c   = coherent_information(rho_RA)
    D_mA  = quantum_discord_wrt_A(rho_RA)
    D_mR  = quantum_discord_wrt_R(rho_RA)
    print(f"\n  Outputs for {state_name}")
    print(f"    I(R:A)          = {I_RA:.6f}  bits")
    print(f"    I_c(R>A)        = {I_c:.6f}  bits")
    print(f"    Discord D(R|A)  = {D_mA:.6f}  bits")
    print(f"    Discord D(A|R)  = {D_mR:.6f}  bits")   # <-- new line
   
    

In [4]:
# ============================================================
# U(4) — KAK DECOMPOSITION (16 real parameters)
# ============================================================

_X = np.array([[0,1],[1,0]],    dtype=complex)
_Y = np.array([[0,-1j],[1j,0]], dtype=complex)
_Z = np.array([[1,0],[0,-1]],   dtype=complex)

def su2(alpha, beta, gamma):
    def Rz(t):
        return np.array([[np.exp(-0.5j*t), 0],
                         [0, np.exp( 0.5j*t)]], dtype=complex)
    def Ry(t):
        c, s = np.cos(t/2), np.sin(t/2)
        return np.array([[c, -s],[s, c]], dtype=complex)
    return Rz(alpha) @ Ry(beta) @ Rz(gamma)

def build_u4(params):
    A1 = su2(*params[0:3]);  A2 = su2(*params[3:6])
    A3 = su2(*params[6:9]);  A4 = su2(*params[9:12])
    c1, c2, c3 = params[12], params[13], params[14]
    phi = params[15]
    XX  = np.kron(_X, _X);  YY = np.kron(_Y, _Y);  ZZ = np.kron(_Z, _Z)
    interaction = expm(1j * (c1*XX + c2*YY + c3*ZZ))
    U = np.kron(A1, A2) @ interaction @ np.kron(A3, A4)
    return U * np.exp(1j * phi)

In [5]:
# ============================================================
# COST FUNCTION
# ============================================================

def apply_u_on_AE(rho_RAE, U):
    U_full = np.kron(np.eye(2, dtype=complex), U)
    return U_full @ rho_RAE @ U_full.conj().T

def cost_function(params, rho_RA):
    U       = build_u4(params)
    ket0_E  = np.array([[1, 0], [0, 0]], dtype=complex)
    rho_RAE = np.kron(rho_RA, ket0_E)
    sigma_RBE = apply_u_on_AE(rho_RAE, U)
    dims = [2, 2, 2]
    sigma_R  = partial_trace_out(sigma_RBE, keep=[0],    dims=dims)
    sigma_B  = partial_trace_out(sigma_RBE, keep=[1],    dims=dims)
    sigma_E  = partial_trace_out(sigma_RBE, keep=[2],    dims=dims)
    sigma_RB = partial_trace_out(sigma_RBE, keep=[0, 1], dims=dims)
    sigma_RE = partial_trace_out(sigma_RBE, keep=[0, 2], dims=dims)
    I_RB = mutual_information(sigma_RB, sigma_R, sigma_B)
    I_RE = mutual_information(sigma_RE, sigma_R, sigma_E)
    return I_RB + I_RE

In [6]:
# ============================================================
# OPTIMIZER
# ============================================================

def find_perfect_hiding(rho_RA, state_name, n_restarts=30):
    print(f"\n{'='*60}")
    print(f"State   : {state_name}")
    best_cost = float('inf')

    
    state_diagnostics(rho_RA, state_name)

    for _ in range(n_restarts):
        x0     = np.random.uniform(-np.pi, np.pi, 16)
        result = minimize(cost_function, x0, args=(rho_RA,),
                          method='L-BFGS-B',
                          bounds=[(-np.pi, np.pi)] * 16,
                          options={'maxiter': 5000, 'ftol': 1e-14, 'gtol': 1e-9})
        if result.fun < best_cost:
            best_cost = result.fun
    achieved = best_cost < 1e-9
    
    print(f"Min I(R:B)+I(R:E) = {best_cost:.9f} ")
    return achieved, best_cost

In [7]:
# ============================================================
# HELPERS
# ============================================================

def ket_to_dm(psi):
    psi = np.array(psi, dtype=complex)
    psi /= np.linalg.norm(psi)
    return np.outer(psi, psi.conj())

def sep_cq(p1, rhoR1, phiA1, p2, rhoR2, phiA2):
    dmA1 = np.outer(phiA1, phiA1.conj())
    dmA2 = np.outer(phiA2, phiA2.conj())
    dmR1 = np.outer(rhoR1, rhoR1.conj()) if rhoR1.ndim == 1 else rhoR1
    dmR2 = np.outer(rhoR2, rhoR2.conj()) if rhoR2.ndim == 1 else rhoR2
    return p1 * np.kron(dmR1, dmA1) + p2 * np.kron(dmR2, dmA2)



ket0   = np.array([1, 0], dtype=complex)
ket1   = np.array([0, 1], dtype=complex)
ket_p  = np.array([1,  1], dtype=complex) / np.sqrt(2)
ket_m  = np.array([1, -1], dtype=complex) / np.sqrt(2)
ket_pi = np.array([1,  1j], dtype=complex) / np.sqrt(2)
ket_mi = np.array([1, -1j], dtype=complex) / np.sqrt(2)
dm0    = ket_to_dm(ket0)
dm1    = ket_to_dm(ket1)

phi_plus     = np.array([1, 0, 0, 1], dtype=complex) / np.sqrt(2)
dm_phi_plus  = ket_to_dm(phi_plus)
psi_minus    = np.array([0, 1, -1, 0], dtype=complex) / np.sqrt(2)
dm_psi_minus = ket_to_dm(psi_minus)
psi_plus     = np.array([0, 1,  1, 0], dtype=complex) / np.sqrt(2)
dm_psi_plus  = ket_to_dm(psi_plus)

---

## Category 1 : Pure Entangled States

---

### PE-1 · Bell State $|\Phi^+\rangle$

**Definition:**

$$|\Phi^+\rangle = \frac{1}{\sqrt{2}}\left(|00\rangle + |11\rangle\right)$$

**Density matrix:**

$$\rho_{PE1} = |\Phi^+\rangle\langle\Phi^+| = \frac{1}{2}\begin{pmatrix}1&0&0&1\\0&0&0&0\\0&0&0&0\\1&0&0&1\end{pmatrix}$$

**Properties:** Maximally entangled. Schmidt rank 2. $S(\rho_R) = S(\rho_A) = 1$ ebit. Real coefficients.

In [8]:
psi_PE1  = np.array([1, 0, 0, 1], dtype=complex) / np.sqrt(2)
rho_PE1  = ket_to_dm(psi_PE1)
find_perfect_hiding(rho_PE1, "PE-1: Bell |Phi+>")


State   : PE-1: Bell |Phi+>

  Outputs for PE-1: Bell |Phi+>
    I(R:A)          = 2.000000  bits
    I_c(R>A)        = 1.000000  bits
    Discord D(R|A)  = 1.000000  bits
    Discord D(A|R)  = 1.000000  bits
Min I(R:B)+I(R:E) = 2.000000000 


(False, 1.9999999999999984)

### PE-2 · Bell State $|\Psi^-\rangle$

**Definition:**

$$|\Psi^-\rangle = \frac{1}{\sqrt{2}}\left(|01\rangle - |10\rangle\right)$$

**Density matrix:**

$$\rho_{PE2} = \frac{1}{2}\begin{pmatrix}0&0&0&0\\0&1&-1&0\\0&-1&1&0\\0&0&0&0\end{pmatrix}$$

**Properties:** Maximally entangled. Antisymmetric singlet state. Real coefficients with relative minus sign.


In [9]:
psi_PE2  = np.array([0, 1, -1, 0], dtype=complex) / np.sqrt(2)
rho_PE2  = ket_to_dm(psi_PE2)
find_perfect_hiding(rho_PE2, "PE-2: Bell |Psi->")


State   : PE-2: Bell |Psi->

  Outputs for PE-2: Bell |Psi->
    I(R:A)          = 2.000000  bits
    I_c(R>A)        = 1.000000  bits
    Discord D(R|A)  = 1.000000  bits
    Discord D(A|R)  = 1.000000  bits
Min I(R:B)+I(R:E) = 2.000000000 


(False, 1.9999999999999984)

### PE-3 · Tilted Bell State

**Definition:**

$$|\psi_{PE3}\rangle = \cos\!\left(\frac{\pi}{3}\right)|00\rangle + \sin\!\left(\frac{\pi}{3}\right)|11\rangle = \frac{1}{2}|00\rangle + \frac{\sqrt{3}}{2}|11\rangle$$

**Density matrix:**

$$\rho_{PE3} = \begin{pmatrix}\frac{1}{4}&0&0&\frac{\sqrt{3}}{4}\\0&0&0&0\\0&0&0&0\\\frac{\sqrt{3}}{4}&0&0&\frac{3}{4}\end{pmatrix}$$

**Schmidt coefficients:** $\lambda_1 = \frac{1}{4},\; \lambda_2 = \frac{3}{4}$. Non-maximally entangled. Real coefficients.

In [10]:
th       = np.pi / 3
psi_PE3  = np.array([np.cos(th), 0, 0, np.sin(th)], dtype=complex)
rho_PE3  = ket_to_dm(psi_PE3)
find_perfect_hiding(rho_PE3, "PE-3: Tilted Bell theta=pi/3")


State   : PE-3: Tilted Bell theta=pi/3

  Outputs for PE-3: Tilted Bell theta=pi/3
    I(R:A)          = 1.622556  bits
    I_c(R>A)        = 0.811278  bits
    Discord D(R|A)  = 0.811278  bits
    Discord D(A|R)  = 0.811278  bits
Min I(R:B)+I(R:E) = 1.622556249 


(False, 1.6225562489182637)

### PE-4 · Complex Phase Bell State

**Definition:**

$$|\psi_{PE4}\rangle = \frac{1}{\sqrt{2}}\left(|00\rangle + i|11\rangle\right)$$

**Density matrix:**

$$\rho_{PE4} = \frac{1}{2}\begin{pmatrix}1&0&0&-i\\0&0&0&0\\0&0&0&0\\i&0&0&1\end{pmatrix}$$

**Properties:** Maximally entangled. Local unitary equivalent to $|\Phi^+\rangle$ via $R_z(\pi/2)$ on one qubit. 

In [11]:
psi_PE4  = np.array([1, 0, 0, 1j], dtype=complex) / np.sqrt(2)
rho_PE4  = ket_to_dm(psi_PE4)
find_perfect_hiding(rho_PE4, "PE-4: (|00>+i|11>)/sqrt(2)")


State   : PE-4: (|00>+i|11>)/sqrt(2)

  Outputs for PE-4: (|00>+i|11>)/sqrt(2)
    I(R:A)          = 2.000000  bits
    I_c(R>A)        = 1.000000  bits
    Discord D(R|A)  = 1.000000  bits
    Discord D(A|R)  = 1.000000  bits
Min I(R:B)+I(R:E) = 2.000000000 


(False, 1.9999999999999978)

### PE-5 · Non-Maximally Entangled with Complex Phase

**Definition:**

$$|\psi_{PE5}\rangle = \sqrt{0.3}\,|00\rangle + \sqrt{0.7}\,e^{i\pi/4}|11\rangle$$

**Density matrix:**

$$\rho_{PE5} = \begin{pmatrix}0.3&0&0&\sqrt{0.21}\,e^{-i\pi/4}\\0&0&0&0\\0&0&0&0\\\sqrt{0.21}\,e^{i\pi/4}&0&0&0.7\end{pmatrix}$$

**Schmidt coefficients:** $\lambda_1 = 0.3,\;\lambda_2 = 0.7$. Combines non-maximal entanglement with a non-trivial complex phase.

In [12]:
psi_PE5  = np.array([np.sqrt(0.3), 0, 0,
                      np.sqrt(0.7) * np.exp(1j * np.pi / 4)], dtype=complex)
rho_PE5  = ket_to_dm(psi_PE5)
find_perfect_hiding(rho_PE5, "PE-5: sqrt(0.3)|00>+sqrt(0.7)e^{ipi/4}|11>")


State   : PE-5: sqrt(0.3)|00>+sqrt(0.7)e^{ipi/4}|11>

  Outputs for PE-5: sqrt(0.3)|00>+sqrt(0.7)e^{ipi/4}|11>
    I(R:A)          = 1.762582  bits
    I_c(R>A)        = 0.881291  bits
    Discord D(R|A)  = 0.881291  bits
    Discord D(A|R)  = 0.881291  bits
Min I(R:B)+I(R:E) = 1.762581798 


(False, 1.7625817984613825)

---

## Category 2 : Mixed Entangled (Non-Separable) States

---

### NM-1 · Werner State $p = 0.8$

**Definition:**

$$\rho_{W}(p) = p\,|\Phi^+\rangle\langle\Phi^+| + \frac{1-p}{4}\,I_4$$

with $p = 0.8$.

**Explicit matrix:**

$$\rho_{NM1} = \frac{1}{4}\begin{pmatrix}1+2p&0&0&4p\\0&1-2p&0&0\\0&0&1-2p&0\\4p&0&0&1+2p\end{pmatrix}\Bigg|_{p=0.8} = \frac{1}{4}\begin{pmatrix}2.6&0&0&3.2\\0&-0.6&0&0\\0&0&-0.6&0\\3.2&0&0&2.6\end{pmatrix}$$

**Separability criterion:** Entangled if and only if $p > \frac{1}{3}$. At $p = 0.8$ it is well into the entangled regime.

**Purity:** $\mathrm{Tr}(\rho^2) = p^2 + \frac{(1-p)^2}{4} + \frac{p(1-p)}{2} < 1$. Genuinely mixed.

In [13]:
phi_plus    = np.array([1, 0, 0, 1], dtype=complex) / np.sqrt(2)
dm_phi_plus = ket_to_dm(phi_plus)

p        = 0.8
rho_NM1  = p * dm_phi_plus + ((1 - p) / 4) * np.eye(4, dtype=complex)
find_perfect_hiding(rho_NM1, "NM-1: Werner p=0.8")


State   : NM-1: Werner p=0.8

  Outputs for NM-1: Werner p=0.8
    I(R:A)          = 1.152415  bits
    I_c(R>A)        = 0.152415  bits
    Discord D(R|A)  = 0.621411  bits
    Discord D(A|R)  = 0.621411  bits
Min I(R:B)+I(R:E) = 1.062008813 


(False, 1.062008812821436)

### NM-2 · Werner State $p = 0.5$ (Entangled but Closer to Boundary)

**Definition:**

$$\rho_{W}(p) = p\,|\Phi^+\rangle\langle\Phi^+| + \frac{1-p}{4}\,I_4$$

with $p = 0.5$.

**Properties:** Still entangled ($p > \frac{1}{3}$) but with weaker correlations than NM-1. Probes how the VQA behaves near the separability boundary.


In [14]:
p        = 0.5
rho_NM2  = p * dm_phi_plus + ((1 - p) / 4) * np.eye(4, dtype=complex)
find_perfect_hiding(rho_NM2, "NM-2: Werner p=0.5")


State   : NM-2: Werner p=0.5

  Outputs for NM-2: Werner p=0.5
    I(R:A)          = 0.451205  bits
    I_c(R>A)        = -0.548795  bits
    Discord D(R|A)  = 0.262483  bits
    Discord D(A|R)  = 0.262483  bits
Min I(R:B)+I(R:E) = 0.377443751 


(False, 0.3774437510817339)

### NM-3 · Isotropic State $f = 0.75$

**Definition:**

$$\rho_{\mathrm{iso}}(f) = f\,|\Phi^+\rangle\langle\Phi^+| + \frac{1-f}{3}\left(I_4 - |\Phi^+\rangle\langle\Phi^+|\right)$$

with $f = 0.75$.

**Separability criterion:** Entangled if and only if $f > \frac{1}{d+1} = \frac{1}{3}$ for $d=2$. At $f=0.75$ it is entangled.

**Relationship to Werner:** The isotropic state is the image of the Werner state under the partial transpose map; they share the same entanglement boundary.

In [15]:
f       = 0.75
rho_NM3 = f * dm_phi_plus + ((1-f)/3) * (np.eye(4, dtype=complex) - dm_phi_plus)
find_perfect_hiding(rho_NM3, "NM-3: Isotropic f=0.75")


State   : NM-3: Isotropic f=0.75

  Outputs for NM-3: Isotropic f=0.75
    I(R:A)          = 0.792481  bits
    I_c(R>A)        = -0.207519  bits
    Discord D(R|A)  = 0.442504  bits
    Discord D(A|R)  = 0.442504  bits
Min I(R:B)+I(R:E) = 0.699955157 


(False, 0.6999551567032924)

### NM-4 · Mixture of Two Bell States

**Definition:**

$$\rho_{NM4} = 0.6\,|\Phi^+\rangle\langle\Phi^+| + 0.4\,|\Psi^-\rangle\langle\Psi^-|$$

**Explicit matrix:**

$$\rho_{NM4} = \frac{1}{2}\begin{pmatrix}0.6&0&0&0.6\\0&0.4&-0.4&0\\0&-0.4&0.4&0\\0.6&0&0&0.6\end{pmatrix}$$

**Properties:** Mixture of two maximally entangled states with real coefficients. Both components are entangled, so the mixture is entangled. Not a Werner state.

In [16]:
psi_minus    = np.array([0, 1, -1, 0], dtype=complex) / np.sqrt(2)
dm_psi_minus = ket_to_dm(psi_minus)

rho_NM4  = 0.6 * dm_phi_plus + 0.4 * dm_psi_minus
find_perfect_hiding(rho_NM4, "NM-4: 0.6|Phi+>+0.4|Psi->")


State   : NM-4: 0.6|Phi+>+0.4|Psi->

  Outputs for NM-4: 0.6|Phi+>+0.4|Psi->
    I(R:A)          = 1.029049  bits
    I_c(R>A)        = 0.029049  bits
    Discord D(R|A)  = 0.029049  bits
    Discord D(A|R)  = 0.029049  bits
Min I(R:B)+I(R:E) = 0.058098811 


(False, 0.05809881109065973)

### NM-5 · Complex Phase Mixture with White Noise

**Definition:**

Let $|\varphi\rangle = \frac{1}{2}\left(|00\rangle + e^{i\pi/3}|01\rangle + e^{i2\pi/3}|10\rangle + |11\rangle\right)$. Then:

$$\rho_{NM5} = 0.7\,|\varphi\rangle\langle\varphi| + 0.3\,\frac{I_4}{4}$$

**Properties:** The pure component $|\varphi\rangle$ is entangled (non-product complex state). White noise depolarises it but not enough to make it separable. Tests the ansatz on complex off-diagonal entries.

In [17]:
phi_c    = np.array([1, np.exp(1j*np.pi/3),
                     np.exp(1j*2*np.pi/3), 1], dtype=complex) / 2.0
rho_NM5  = 0.7 * ket_to_dm(phi_c) + 0.3 * np.eye(4, dtype=complex) / 4
find_perfect_hiding(rho_NM5, "NM-5: Complex phase + white noise")


State   : NM-5: Complex phase + white noise

  Outputs for NM-5: Complex phase + white noise
    I(R:A)          = 0.874191  bits
    I_c(R>A)        = -0.125809  bits
    Discord D(R|A)  = 0.484031  bits
    Discord D(A|R)  = 0.484031  bits
Min I(R:B)+I(R:E) = 0.780319391 


(False, 0.7803193905671975)

---

## Category 3A : Separable States class 1
---

### AC-1 · Hadamard-Rotated Classical Mixture

**Construction:** Start from the perfectly classical diagonal state $\frac{1}{2}|00\rangle\langle 00| + \frac{1}{2}|11\rangle\langle 11|$ and apply $H_R \otimes I_A$:

$$\rho_{AC1} = \frac{1}{2}|{+}\rangle\langle{+}|_R \otimes |0\rangle\langle 0|_A + \frac{1}{2}|{-}\rangle\langle{-}|_R \otimes |1\rangle\langle 1|_A$$

**Explicit matrix in computational basis:**

$$\rho_{AC1} = \frac{1}{4}\begin{pmatrix}1&0&1&0\\0&1&0&-1\\1&0&1&0\\0&-1&0&1\end{pmatrix}$$

**Why apparent only:** Apply $H_R \otimes I_A$ and you recover $\frac{1}{2}|00\rangle\langle 00| + \frac{1}{2}|11\rangle\langle 11|$ — a zero-discord classical state. The off-diagonals are entirely due to the basis rotation.


In [18]:
rho_AC1  = sep_cq(0.5, ket_p, ket0, 0.5, ket_m, ket1)
find_perfect_hiding(rho_AC1, "AC-1: Hadamard-rotated classical mixture")


State   : AC-1: Hadamard-rotated classical mixture

  Outputs for AC-1: Hadamard-rotated classical mixture
    I(R:A)          = 1.000000  bits
    I_c(R>A)        = 0.000000  bits
    Discord D(R|A)  = 0.000000  bits
    Discord D(A|R)  = 0.000000  bits
Min I(R:B)+I(R:E) = -0.000000000 


(True, -8.881784197001252e-16)

### AC-2 · Y-Rotated Classical Mixture

**Construction:** Apply $R_y(\pi/2)_R \otimes I_A$ to $\frac{1}{2}|00\rangle\langle 00| + \frac{1}{2}|11\rangle\langle 11|$:

$$\rho_{AC2} = \frac{1}{2}|{+_i}\rangle\langle{+_i}|_R \otimes |0\rangle\langle 0|_A + \frac{1}{2}|{-_i}\rangle\langle{-_i}|_R \otimes |1\rangle\langle 1|_A$$

where $|{\pm_i}\rangle = \frac{1}{\sqrt{2}}(|0\rangle \pm i|1\rangle)$.

**Why apparent only:** The $Y$-eigenstates on $R$ are related to $|0\rangle, |1\rangle$ by a single-qubit unitary. Under $U_R^\dagger \otimes I_A$ the state becomes diagonal. Complex off-diagonals in the computational basis but zero discord.


In [19]:
rho_AC2  = sep_cq(0.5, ket0, ket_pi, 0.5, ket1, ket_mi)
find_perfect_hiding(rho_AC2, "AC-2: Y-rotated classical mixture (complex R)")


State   : AC-2: Y-rotated classical mixture (complex R)

  Outputs for AC-2: Y-rotated classical mixture (complex R)
    I(R:A)          = 1.000000  bits
    I_c(R>A)        = 0.000000  bits
    Discord D(R|A)  = 0.000000  bits
    Discord D(A|R)  = 0.000000  bits
Min I(R:B)+I(R:E) = 0.000000000 


(True, 2.220446049250313e-16)

### AC-3 · Z–X Correlation (Classical in Rotated Basis)

**Definition:**

$$\rho_{AC3} = \frac{1}{2}|0\rangle\langle 0|_R \otimes |{+}\rangle\langle{+}|_A + \frac{1}{2}|1\rangle\langle 1|_R \otimes |{-}\rangle\langle{-}|_A$$

**Explicit matrix:**

$$\rho_{AC3} = \frac{1}{4}\begin{pmatrix}1&1&0&0\\1&1&0&0\\0&0&1&-1\\0&0&-1&1\end{pmatrix}$$

**Why apparent only:** Apply $I_R \otimes H_A$ to recover $\frac{1}{2}|00\rangle\langle 00| + \frac{1}{2}|11\rangle\langle 11|$. The coherence lives entirely in the $A$ subsystem and is removable by a local unitary on $A$.

In [20]:
rho_AC3  = sep_cq(0.5, ket0, ket_p, 0.5, ket1, ket_m)
find_perfect_hiding(rho_AC3, "AC-3: Z-X correlation (apparent coherence on A)")


State   : AC-3: Z-X correlation (apparent coherence on A)

  Outputs for AC-3: Z-X correlation (apparent coherence on A)
    I(R:A)          = 1.000000  bits
    I_c(R>A)        = 0.000000  bits
    Discord D(R|A)  = 0.000000  bits
    Discord D(A|R)  = 0.000000  bits
Min I(R:B)+I(R:E) = -0.000000000 


(True, -2.4424906541753444e-15)

### AC-4 · Asymmetric Weights, Apparent Coherence

**Definition:**

$$\rho_{AC4} = 0.3\,|0\rangle\langle 0|_R \otimes |{+_i}\rangle\langle{+_i}|_A + 0.7\,|1\rangle\langle 1|_R \otimes |{-_i}\rangle\langle{-_i}|_A$$

**Why apparent only:** Same reasoning as AC-2 but with unequal weights. The local unitary $I_R \otimes U_A$ that maps $|{\pm_i}\rangle \to |0\rangle, |1\rangle$ diagonalises the full state. Zero discord.

In [21]:
rho_AC4  = sep_cq(0.3, ket0, ket_pi, 0.7, ket1, ket_mi)
find_perfect_hiding(rho_AC4, "AC-4: Asymmetric apparent coherence")


State   : AC-4: Asymmetric apparent coherence

  Outputs for AC-4: Asymmetric apparent coherence
    I(R:A)          = 0.881291  bits
    I_c(R>A)        = 0.000000  bits
    Discord D(R|A)  = 0.000000  bits
    Discord D(A|R)  = 0.000000  bits
Min I(R:B)+I(R:E) = 0.000000000 


(True, 1.1102230246251565e-15)

### AC-5 · Both Subsystems Rotated

**Definition:**

$$\rho_{AC5} = \frac{1}{2}|{+}\rangle\langle{+}|_R \otimes |{+_i}\rangle\langle{+_i}|_A + \frac{1}{2}|{-}\rangle\langle{-}|_R \otimes |{-_i}\rangle\langle{-_i}|_A$$

**Why apparent only:** Apply $H_R \otimes U_A$ simultaneously. This maps $|{\pm}\rangle_R \to |0\rangle,|1\rangle$ and $|{\pm_i}\rangle_A \to |0\rangle,|1\rangle$, recovering a diagonal classical mixture. Both subsystems carry only apparent coherence.

In [22]:
rho_AC5  = sep_cq(0.5, ket_p, ket_pi, 0.5, ket_m, ket_mi)
find_perfect_hiding(rho_AC5, "AC-5: Both subsystems rotated (apparent coherence)")


State   : AC-5: Both subsystems rotated (apparent coherence)

  Outputs for AC-5: Both subsystems rotated (apparent coherence)
    I(R:A)          = 1.000000  bits
    I_c(R>A)        = 0.000000  bits
    Discord D(R|A)  = 0.000000  bits
    Discord D(A|R)  = 0.000000  bits
Min I(R:B)+I(R:E) = -0.000000000 


(True, -1.1102230246251565e-15)

---

## Category 3B : Separable States  class 2

---

### TD-1 · Z–X CQ State (Non-Orthogonal A in Any Z-Measurement Basis)

**Definition:**

$$\rho_{TD1} = \frac{1}{2}|0\rangle\langle 0|_R \otimes |{+}\rangle\langle{+}|_A + \frac{1}{2}|1\rangle\langle 1|_R \otimes |{-}\rangle\langle{-}|_A$$



In [23]:
rho_TD1  = sep_cq(0.5, ket0, ket_p, 0.5, ket1, ket_m)
find_perfect_hiding(rho_TD1, "TD-1: True discord, Z-X CQ, pure A")


State   : TD-1: True discord, Z-X CQ, pure A

  Outputs for TD-1: True discord, Z-X CQ, pure A
    I(R:A)          = 1.000000  bits
    I_c(R>A)        = 0.000000  bits
    Discord D(R|A)  = 0.000000  bits
    Discord D(A|R)  = 0.000000  bits
Min I(R:B)+I(R:E) = -0.000000000 


(True, -3.3306690738754696e-15)

### TD-2 · Asymmetric CQ with Complex Pure A

**Definition:**

$$\rho_{TD2} = \frac{1}{3}|0\rangle\langle 0|_R \otimes |\phi_1\rangle\langle\phi_1|_A + \frac{2}{3}|1\rangle\langle 1|_R \otimes |\phi_2\rangle\langle\phi_2|_A$$

where:

$$|\phi_1\rangle = \cos\!\left(\frac{\pi}{8}\right)|0\rangle + e^{i\pi/3}\sin\!\left(\frac{\pi}{8}\right)|1\rangle, \qquad |\phi_2\rangle = \cos\!\left(\frac{3\pi}{8}\right)|0\rangle + e^{-i\pi/4}\sin\!\left(\frac{3\pi}{8}\right)|1\rangle$$


In [24]:
phi_td2_1 = np.array([np.cos(np.pi/8),
                       np.exp(1j*np.pi/3) * np.sin(np.pi/8)], dtype=complex)
phi_td2_2 = np.array([np.cos(3*np.pi/8),
                       np.exp(-1j*np.pi/4) * np.sin(3*np.pi/8)], dtype=complex)
rho_TD2   = sep_cq(1/3, ket0, phi_td2_1, 2/3, ket1, phi_td2_2)
find_perfect_hiding(rho_TD2, "TD-2: Asymmetric CQ, complex non-orthogonal pure A")


State   : TD-2: Asymmetric CQ, complex non-orthogonal pure A

  Outputs for TD-2: Asymmetric CQ, complex non-orthogonal pure A
    I(R:A)          = 0.790703  bits
    I_c(R>A)        = -0.127592  bits
    Discord D(R|A)  = 0.128415  bits
    Discord D(A|R)  = 0.000000  bits
Min I(R:B)+I(R:E) = -0.000000000 


(True, -4.440892098500626e-16)

### TD-3 · Trisector A-States (Three Equatorial Pure States)

**Definition:**

$$|\phi_k\rangle = \frac{1}{\sqrt{2}}\left(|0\rangle + e^{i2\pi k/3}|1\rangle\right), \quad k = 0, 1, 2$$

$$\rho_{TD3} = \frac{1}{3}\,|0\rangle\langle 0|_R \otimes |\phi_0\rangle\langle\phi_0|_A + \frac{1}{3}\,|1\rangle\langle 1|_R \otimes |\phi_1\rangle\langle\phi_1|_A + \frac{1}{3}\,\frac{I_R}{2} \otimes |\phi_2\rangle\langle\phi_2|_A$$



In [25]:
phi_td3_0 = np.array([1, np.exp(0j)],             dtype=complex) / np.sqrt(2)
phi_td3_1 = np.array([1, np.exp(1j*2*np.pi/3)],   dtype=complex) / np.sqrt(2)
phi_td3_2 = np.array([1, np.exp(1j*4*np.pi/3)],   dtype=complex) / np.sqrt(2)
rho_TD3   = (1/3) * np.kron(dm0, ket_to_dm(phi_td3_0)) \
          + (1/3) * np.kron(dm1, ket_to_dm(phi_td3_1)) \
          + (1/3) * np.kron(np.eye(2)/2, ket_to_dm(phi_td3_2))
find_perfect_hiding(rho_TD3, "TD-3: Trisector pure A-states (true discord)")


State   : TD-3: Trisector pure A-states (true discord)

  Outputs for TD-3: Trisector pure A-states (true discord)
    I(R:A)          = 0.255992  bits
    I_c(R>A)        = -0.744008  bits
    Discord D(R|A)  = 0.000000  bits
    Discord D(A|R)  = 0.000000  bits
Min I(R:B)+I(R:E) = 0.000000000 


(True, 2.220446049250313e-16)

### TD-4 · Superposition R with Complex Non-Orthogonal A

**Definition:**

$$\rho_{TD4} = \frac{1}{2}|{+_i}\rangle\langle{+_i}|_R \otimes |\phi_1\rangle\langle\phi_1|_A + \frac{1}{2}|{-_i}\rangle\langle{-_i}|_R \otimes |\phi_2\rangle\langle\phi_2|_A$$

where:

$$|\phi_1\rangle = \frac{1}{\sqrt{2}}\begin{pmatrix}1\\e^{i\pi/6}\end{pmatrix}, \qquad |\phi_2\rangle = \frac{1}{\sqrt{2}}\begin{pmatrix}1\\e^{-i\pi/6}\end{pmatrix}$$



In [26]:
phi_td4_1 = np.array([1, np.exp( 1j*np.pi/6)], dtype=complex) / np.sqrt(2)
phi_td4_2 = np.array([1, np.exp(-1j*np.pi/6)], dtype=complex) / np.sqrt(2)
rho_TD4   = sep_cq(0.5, ket_pi, phi_td4_1, 0.5, ket_mi, phi_td4_2)
find_perfect_hiding(rho_TD4, "TD-4: Complex R and non-orthogonal A (true discord)")


State   : TD-4: Complex R and non-orthogonal A (true discord)

  Outputs for TD-4: Complex R and non-orthogonal A (true discord)
    I(R:A)          = 0.354579  bits
    I_c(R>A)        = -0.645421  bits
    Discord D(R|A)  = 0.165857  bits
    Discord D(A|R)  = 0.000000  bits
Min I(R:B)+I(R:E) = 0.000000000 


(True, 0.0)

### TD-5 · Werner-Like Separable State at the Discord Boundary

**Definition:**

The separable Werner state sits exactly at $p = \frac{1}{3}$:

$$\rho_{W}\!\left(\tfrac{1}{3}\right) = \frac{1}{3}\,|\Phi^+\rangle\langle\Phi^+| + \frac{1}{6}\,I_4$$


In [27]:
p_boundary = 1/3
rho_TD5    = p_boundary * dm_phi_plus + (1 - p_boundary) / 4 * np.eye(4, dtype=complex)
find_perfect_hiding(rho_TD5, "TD-5: Werner at separability boundary (discord, mixed A)")


State   : TD-5: Werner at separability boundary (discord, mixed A)

  Outputs for TD-5: Werner at separability boundary (discord, mixed A)
    I(R:A)          = 0.207519  bits
    I_c(R>A)        = -0.792481  bits
    Discord D(R|A)  = 0.125815  bits
    Discord D(A|R)  = 0.125815  bits
Min I(R:B)+I(R:E) = 0.163408332 


(False, 0.16340833189102)

### TD-5' · Werner-Like Separable State at the Discord Boundary

**Definition:**

The separable Werner state sits exactly at $p = \frac{1}{4}$:

$$\rho_{W}\!\left(\tfrac{1}{3}\right) = \frac{1}{3}\,|\Phi^+\rangle\langle\Phi^+| + \frac{3}{16}\,I_4$$

In [28]:
p = 1/4
rho_TD5    = p * dm_phi_plus + (1 - p) / 4 * np.eye(4, dtype=complex)
find_perfect_hiding(rho_TD5, "TD-5': Werner at separability boundary (discord, mixed A)")


State   : TD-5': Werner at separability boundary (discord, mixed A)

  Outputs for TD-5': Werner at separability boundary (discord, mixed A)
    I(R:A)          = 0.119759  bits
    I_c(R>A)        = -0.880241  bits
    Discord D(R|A)  = 0.074193  bits
    Discord D(A|R)  = 0.074193  bits
Min I(R:B)+I(R:E) = 0.091131994 


(False, 0.09113199415007123)

---

## Category 4 : Separable States with Non-Pure A-Components

---

### SM-1 · Fully Mixed Product State

**Definition:**

$$\rho_{SM1} = \frac{I_R}{2} \otimes \frac{I_A}{2} = \frac{I_4}{4}$$

**Properties:** Zero entanglement, zero discord, zero correlations of any kind. The A-marginal is $I/2$, which is maximally mixed — the furthest possible from a pure state.


In [29]:
rho_SM1  = np.eye(4, dtype=complex) / 4
find_perfect_hiding(rho_SM1, "SM-1: I/2 x I/2 (fully mixed product)")


State   : SM-1: I/2 x I/2 (fully mixed product)

  Outputs for SM-1: I/2 x I/2 (fully mixed product)
    I(R:A)          = 0.000000  bits
    I_c(R>A)        = -1.000000  bits
    Discord D(R|A)  = 0.000000  bits
    Discord D(A|R)  = 0.000000  bits
Min I(R:B)+I(R:E) = -0.000000000 


(True, -4.440892098500626e-15)

### SM-2 · Pure R, Maximally Mixed A

**Definition:**

$$\rho_{SM2} = |{+}\rangle\langle{+}|_R \otimes \frac{I_A}{2}$$



In [30]:
rho_SM2  = np.kron(ket_to_dm(ket_p), np.eye(2, dtype=complex) / 2)
find_perfect_hiding(rho_SM2, "SM-2: |+><+|_R x I/2_A")


State   : SM-2: |+><+|_R x I/2_A

  Outputs for SM-2: |+><+|_R x I/2_A
    I(R:A)          = -0.000000  bits
    I_c(R>A)        = 0.000000  bits
    Discord D(R|A)  = 0.000000  bits
    Discord D(A|R)  = 0.000000  bits
Min I(R:B)+I(R:E) = -0.000000000 


(True, -3.9968028886505635e-15)

### SM-3 · CQ State, Mixed A with Real Off-Diagonals

**Definition:**

$$\rho_{SM3} = \frac{1}{2}|0\rangle\langle 0|_R \otimes \rho_A^{(1)} + \frac{1}{2}|1\rangle\langle 1|_R \otimes \rho_A^{(2)}$$

where:

$$\rho_A^{(1)} = \begin{pmatrix}0.7 & 0.1\\0.1 & 0.3\end{pmatrix}, \qquad \rho_A^{(2)} = \begin{pmatrix}0.4 & -0.1\\-0.1 & 0.6\end{pmatrix}$$



In [31]:
rhoA_sm3_1 = np.array([[0.7,  0.1], [ 0.1, 0.3]], dtype=complex)
rhoA_sm3_2 = np.array([[0.4, -0.1], [-0.1, 0.6]], dtype=complex)
rho_SM3    = 0.5 * np.kron(dm0, rhoA_sm3_1) + 0.5 * np.kron(dm1, rhoA_sm3_2)
find_perfect_hiding(rho_SM3, "SM-3: CQ, mixed A, real off-diag")


State   : SM-3: CQ, mixed A, real off-diag

  Outputs for SM-3: CQ, mixed A, real off-diag
    I(R:A)          = 0.096781  bits
    I_c(R>A)        = -0.903219  bits
    Discord D(R|A)  = 0.000105  bits
    Discord D(A|R)  = 0.000000  bits
Min I(R:B)+I(R:E) = -0.000000000 


(True, -1.7763568394002505e-15)

### SM-4 · CQ State, Mixed A with Complex Off-Diagonals

**Definition:**

$$\rho_{SM4} = \frac{1}{2}|0\rangle\langle 0|_R \otimes \rho_A^{(1)} + \frac{1}{2}|1\rangle\langle 1|_R \otimes \rho_A^{(2)}$$

where:

$$\rho_A^{(1)} = \begin{pmatrix}0.6 & 0.1i\\-0.1i & 0.4\end{pmatrix}, \qquad \rho_A^{(2)} = \begin{pmatrix}0.5 & -0.2i\\0.2i & 0.5\end{pmatrix}$$



In [32]:
rhoA_sm4_1 = np.array([[0.6,  0.1j], [-0.1j, 0.4]], dtype=complex)
rhoA_sm4_2 = np.array([[0.5, -0.2j], [ 0.2j, 0.5]], dtype=complex)
rho_SM4    = 0.5 * np.kron(dm0, rhoA_sm4_1) + 0.5 * np.kron(dm1, rhoA_sm4_2)
find_perfect_hiding(rho_SM4, "SM-4: CQ, mixed A, complex off-diag")


State   : SM-4: CQ, mixed A, complex off-diag

  Outputs for SM-4: CQ, mixed A, complex off-diag
    I(R:A)          = 0.074131  bits
    I_c(R>A)        = -0.925869  bits
    Discord D(R|A)  = 0.000415  bits
    Discord D(A|R)  = 0.000000  bits
Min I(R:B)+I(R:E) = 0.000000000 


(True, 1.7763568394002505e-15)

### SM-5 · Complex R and A, Convex Mixture

**Definition:**

$$\rho_{SM5} = 0.4\,|{+_i}\rangle\langle{+_i}|_R \otimes \rho_A^{(1)} + 0.6\,|{-_i}\rangle\langle{-_i}|_R \otimes \rho_A^{(2)}$$

where:

$$\rho_A^{(1)} = \begin{pmatrix}0.8 & 0.1+0.1i\\0.1-0.1i & 0.2\end{pmatrix}, \qquad \rho_A^{(2)} = \begin{pmatrix}0.3 & -0.1+0.05i\\-0.1-0.05i & 0.7\end{pmatrix}$$


In [33]:
rhoA_sm5_1 = np.array([[0.8,  0.1+0.1j],  [ 0.1-0.1j,  0.2]], dtype=complex)
rhoA_sm5_2 = np.array([[0.3, -0.1+0.05j], [-0.1-0.05j, 0.7]], dtype=complex)
rho_SM5    = 0.4 * np.kron(ket_to_dm(ket_pi), rhoA_sm5_1) \
           + 0.6 * np.kron(ket_to_dm(ket_mi), rhoA_sm5_2)
find_perfect_hiding(rho_SM5, "SM-5: Complex R and A, convex mixture")


State   : SM-5: Complex R and A, convex mixture

  Outputs for SM-5: Complex R and A, convex mixture
    I(R:A)          = 0.217471  bits
    I_c(R>A)        = -0.753480  bits
    Discord D(R|A)  = 0.001801  bits
    Discord D(A|R)  = 0.000000  bits
Min I(R:B)+I(R:E) = -0.000000000 


(True, -6.661338147750939e-16)

---

## Category 5 : Separable States with Pure A-Components

---

### SP-1 · Orthogonal Pure A in Computational Basis

**Definition:**

$$\rho_{SP1} = \frac{1}{2}|0\rangle\langle 0|_R \otimes |0\rangle\langle 0|_A + \frac{1}{2}|1\rangle\langle 1|_R \otimes |1\rangle\langle 1|_A$$

$$\rho_{SP1} = \frac{1}{2}\begin{pmatrix}1&0&0&0\\0&0&0&0\\0&0&0&0\\0&0&0&1\end{pmatrix}$$


In [34]:
rho_SP1  = sep_cq(0.5, ket0, ket0, 0.5, ket1, ket1)
find_perfect_hiding(rho_SP1, "SP-1: CQ, orthogonal pure A (Z-basis)")


State   : SP-1: CQ, orthogonal pure A (Z-basis)

  Outputs for SP-1: CQ, orthogonal pure A (Z-basis)
    I(R:A)          = 1.000000  bits
    I_c(R>A)        = 0.000000  bits
    Discord D(R|A)  = 0.000000  bits
    Discord D(A|R)  = 0.000000  bits
Min I(R:B)+I(R:E) = -0.000000000 


(True, -4.440892098500626e-16)

### SP-2 · Non-Orthogonal Pure A in X-Basis (Real)

**Definition:**

$$\rho_{SP2} = \frac{1}{2}|0\rangle\langle 0|_R \otimes |{+}\rangle\langle{+}|_A + \frac{1}{2}|1\rangle\langle 1|_R \otimes |{-}\rangle\langle{-}|_A$$



In [35]:
rho_SP2  = sep_cq(0.5, ket0, ket_p, 0.5, ket1, ket_m)
find_perfect_hiding(rho_SP2, "SP-2: CQ, non-orthogonal pure A (X-basis)")


State   : SP-2: CQ, non-orthogonal pure A (X-basis)

  Outputs for SP-2: CQ, non-orthogonal pure A (X-basis)
    I(R:A)          = 1.000000  bits
    I_c(R>A)        = 0.000000  bits
    Discord D(R|A)  = 0.000000  bits
    Discord D(A|R)  = 0.000000  bits
Min I(R:B)+I(R:E) = -0.000000000 


(True, -2.6645352591003757e-15)

### SP-3 · Pure A in Y-Basis (Complex)

**Definition:**

$$\rho_{SP3} = \frac{1}{2}|0\rangle\langle 0|_R \otimes |{+_i}\rangle\langle{+_i}|_A + \frac{1}{2}|1\rangle\langle 1|_R \otimes |{-_i}\rangle\langle{-_i}|_A$$

where $$|{\pm_i}\rangle = \frac{1}{\sqrt{2}}(|0\rangle \pm i|1\rangle)$$.



In [36]:
rho_SP3  = sep_cq(0.5, ket0, ket_pi, 0.5, ket1, ket_mi)
find_perfect_hiding(rho_SP3, "SP-3: CQ, Y-basis pure A (complex)")


State   : SP-3: CQ, Y-basis pure A (complex)

  Outputs for SP-3: CQ, Y-basis pure A (complex)
    I(R:A)          = 1.000000  bits
    I_c(R>A)        = 0.000000  bits
    Discord D(R|A)  = 0.000000  bits
    Discord D(A|R)  = 0.000000  bits
Min I(R:B)+I(R:E) = -0.000000000 


(True, -4.440892098500626e-16)

### SP-4 · Asymmetric Weights, Complex Pure A

**Definition:**

$$\rho_{SP4} = \frac{1}{3}|0\rangle\langle 0|_R \otimes |\phi_1\rangle\langle\phi_1|_A + \frac{2}{3}|1\rangle\langle 1|_R \otimes |\phi_2\rangle\langle\phi_2|_A$$

where:

$$|\phi_1\rangle = \cos\!\left(\frac{\pi}{8}\right)|0\rangle + e^{i\pi/3}\sin\!\left(\frac{\pi}{8}\right)|1\rangle, \qquad |\phi_2\rangle = \cos\!\left(\frac{3\pi}{8}\right)|0\rangle + e^{-i\pi/4}\sin\!\left(\frac{3\pi}{8}\right)|1\rangle$$


In [37]:
phi_sp4_1 = np.array([np.cos(np.pi/8),
                       np.exp( 1j*np.pi/3)*np.sin(np.pi/8)], dtype=complex)
phi_sp4_2 = np.array([np.cos(3*np.pi/8),
                       np.exp(-1j*np.pi/4)*np.sin(3*np.pi/8)], dtype=complex)
rho_SP4   = sep_cq(1/3, ket0, phi_sp4_1, 2/3, ket1, phi_sp4_2)
find_perfect_hiding(rho_SP4, "SP-4: Asymmetric CQ, complex pure A")


State   : SP-4: Asymmetric CQ, complex pure A

  Outputs for SP-4: Asymmetric CQ, complex pure A
    I(R:A)          = 0.790703  bits
    I_c(R>A)        = -0.127592  bits
    Discord D(R|A)  = 0.128415  bits
    Discord D(A|R)  = 0.000000  bits
Min I(R:B)+I(R:E) = -0.000000000 


(True, -6.661338147750939e-16)

### SP-5 · Complex R and Complex Pure A

**Definition:**

$$\rho_{SP5} = \frac{1}{2}|{+_i}\rangle\langle{+_i}|_R \otimes |\phi_1\rangle\langle\phi_1|_A + \frac{1}{2}|{-_i}\rangle\langle{-_i}|_R \otimes |\phi_2\rangle\langle\phi_2|_A$$

where:

$$|\phi_1\rangle = \frac{1}{\sqrt{2}}\begin{pmatrix}1\\e^{i\pi/6}\end{pmatrix}, \qquad |\phi_2\rangle = \frac{1}{\sqrt{2}}\begin{pmatrix}1\\e^{-i\pi/6}\end{pmatrix}$$


In [38]:
phi_sp5_1 = np.array([1, np.exp( 1j*np.pi/6)], dtype=complex) / np.sqrt(2)
phi_sp5_2 = np.array([1, np.exp(-1j*np.pi/6)], dtype=complex) / np.sqrt(2)
rho_SP5   = sep_cq(0.5, ket_pi, phi_sp5_1, 0.5, ket_mi, phi_sp5_2)
find_perfect_hiding(rho_SP5, "SP-5: Complex R and complex pure A")


State   : SP-5: Complex R and complex pure A

  Outputs for SP-5: Complex R and complex pure A
    I(R:A)          = 0.354579  bits
    I_c(R>A)        = -0.645421  bits
    Discord D(R|A)  = 0.165857  bits
    Discord D(A|R)  = 0.000000  bits
Min I(R:B)+I(R:E) = 0.000000000 


(True, 4.440892098500626e-16)

---
## Category 6 : True Quantum Discord with Non-Pure A-Components
---

### TD6-1 · Non-Commuting Mixed A-Components (Real)

**Definition:**

$$\rho_{TD6-1} = \frac{1}{2}\,|0\rangle\langle 0|_R \otimes \rho_A^{(1)} + \frac{1}{2}\,|1\rangle\langle 1|_R \otimes \rho_A^{(2)}$$

where:

$$\rho_A^{(1)} = \begin{pmatrix}0.85 & 0.2\\0.2 & 0.15\end{pmatrix}, \qquad \rho_A^{(2)} = \begin{pmatrix}0.15 & 0.2\\0.2 & 0.85\end{pmatrix}$$




In [39]:
rhoA_td6_1_1 = np.array([[0.85, 0.2 ], [0.2,  0.15]], dtype=complex)
rhoA_td6_1_2 = np.array([[0.15, 0.2 ], [0.2,  0.85]], dtype=complex)

rho_TD6_1 = (0.5 * np.kron(dm0, rhoA_td6_1_1)
           + 0.5 * np.kron(dm1, rhoA_td6_1_2))
find_perfect_hiding(rho_TD6_1, "TD6-1: Non-commuting mixed A (real)")


State   : TD6-1: Non-commuting mixed A (real)

  Outputs for TD6-1: Non-commuting mixed A (real)
    I(R:A)          = 0.422241  bits
    I_c(R>A)        = -0.577759  bits
    Discord D(R|A)  = 0.032082  bits
    Discord D(A|R)  = 0.000000  bits
Min I(R:B)+I(R:E) = 0.000000000 


(True, 2.220446049250313e-16)

### TD6-2 · Non-Commuting Mixed A-Components (Complex)

**Definition:**

$$\rho_{TD6-2} = \frac{1}{2}\,|0\rangle\langle 0|_R \otimes \rho_A^{(1)} + \frac{1}{2}\,|1\rangle\langle 1|_R \otimes \rho_A^{(2)}$$

where:

$$\rho_A^{(1)} = \begin{pmatrix}0.7 & 0.2i\\-0.2i & 0.3\end{pmatrix}, \qquad \rho_A^{(2)} = \begin{pmatrix}0.6 & 0.3\\0.3 & 0.4\end{pmatrix}$$



In [40]:
rhoA_td6_2_1 = np.array([[0.7,  0.2j], [-0.2j, 0.3]], dtype=complex)
rhoA_td6_2_2 = np.array([[0.6,  0.3 ], [ 0.3,  0.4]], dtype=complex)
rho_TD6_2 = (0.5 * np.kron(dm0, rhoA_td6_2_1)
           + 0.5 * np.kron(dm1, rhoA_td6_2_2))
find_perfect_hiding(rho_TD6_2, "TD6-2: Non-commuting mixed A (X vs Y coherence)")


State   : TD6-2: Non-commuting mixed A (X vs Y coherence)

  Outputs for TD6-2: Non-commuting mixed A (X vs Y coherence)
    I(R:A)          = 0.113301  bits
    I_c(R>A)        = -0.886699  bits
    Discord D(R|A)  = 0.009369  bits
    Discord D(A|R)  = 0.000000  bits
Min I(R:B)+I(R:E) = 0.000000000 


(True, 6.661338147750939e-16)

### TD6-3 · Asymmetric Weights, Non-Commuting Mixed A

**Definition:**

$$\rho_{TD6-3} = \frac{1}{3}\,|0\rangle\langle 0|_R \otimes \rho_A^{(1)} + \frac{2}{3}\,|1\rangle\langle 1|_R \otimes \rho_A^{(2)}$$

where:

$$\rho_A^{(1)} = \frac{1}{2}\begin{pmatrix}1+r\cos\theta & r\sin\theta\\r\sin\theta & 1-r\cos\theta\end{pmatrix}\Bigg|_{r=0.6,\,\theta=\pi/4} = \begin{pmatrix}0.5+\frac{0.6}{\sqrt{2}} & \frac{0.6}{\sqrt{2}}\\\frac{0.6}{\sqrt{2}} & 0.5-\frac{0.6}{\sqrt{2}}\end{pmatrix}$$

$$\rho_A^{(2)} = \frac{1}{2}\begin{pmatrix}1+r\sin\theta & ir\cos\theta\\-ir\cos\theta & 1-r\sin\theta\end{pmatrix}\Bigg|_{r=0.5,\,\theta=\pi/3} = \begin{pmatrix}0.5+\frac{0.5\sqrt{3}}{2} & \frac{0.5i}{2}\\-\frac{0.5i}{2} & 0.5-\frac{0.5\sqrt{3}}{2}\end{pmatrix}$$


In [41]:
r1, theta1 = 0.6, np.pi/4
r2, theta2 = 0.5, np.pi/3

rhoA_td6_3_1 = np.array([
    [0.5 + r1*np.cos(theta1)/2,  r1*np.sin(theta1)/2],
    [r1*np.sin(theta1)/2,         0.5 - r1*np.cos(theta1)/2]
], dtype=complex)

rhoA_td6_3_2 = np.array([
    [0.5 + r2*np.sin(theta2)/2,   1j*r2*np.cos(theta2)/2],
    [-1j*r2*np.cos(theta2)/2,     0.5 - r2*np.sin(theta2)/2]
], dtype=complex)


rho_TD6_3 = (1/3 * np.kron(dm0, rhoA_td6_3_1)
           + 2/3 * np.kron(dm1, rhoA_td6_3_2))
find_perfect_hiding(rho_TD6_3, "TD6-3: Asymmetric weights, Bloch XZ vs YZ plane A")


State   : TD6-3: Asymmetric weights, Bloch XZ vs YZ plane A

  Outputs for TD6-3: Asymmetric weights, Bloch XZ vs YZ plane A
    I(R:A)          = 0.043395  bits
    I_c(R>A)        = -0.874901  bits
    Discord D(R|A)  = 0.003692  bits
    Discord D(A|R)  = 0.000000  bits
Min I(R:B)+I(R:E) = 0.000000000 


(True, 5.329070518200751e-15)

### TD6-4 · Complex R-Components, Non-Commuting Mixed A

**Definition:**

$$\rho_{TD6-4} = \frac{1}{2}\,|{+_i}\rangle\langle{+_i}|_R \otimes \rho_A^{(1)} + \frac{1}{2}\,|{-_i}\rangle\langle{-_i}|_R \otimes \rho_A^{(2)}$$

where:

$$\rho_A^{(1)} = \begin{pmatrix}0.75 & 0.15+0.1i\\0.15-0.1i & 0.25\end{pmatrix}, \qquad \rho_A^{(2)} = \begin{pmatrix}0.4 & -0.2+0.15i\\-0.2-0.15i & 0.6\end{pmatrix}$$



In [42]:
rhoA_td6_4_1 = np.array([[0.75,  0.15+0.1j],  [0.15-0.1j,  0.25]], dtype=complex)
rhoA_td6_4_2 = np.array([[0.4,  -0.2+0.15j],  [-0.2-0.15j, 0.6 ]], dtype=complex)

rho_TD6_4 = (0.5 * np.kron(ket_to_dm(ket_pi), rhoA_td6_4_1)
           + 0.5 * np.kron(ket_to_dm(ket_mi), rhoA_td6_4_2))
find_perfect_hiding(rho_TD6_4, "TD6-4: Complex R, non-commuting mixed A (full complexity)")


State   : TD6-4: Complex R, non-commuting mixed A (full complexity)

  Outputs for TD6-4: Complex R, non-commuting mixed A (full complexity)
    I(R:A)          = 0.193687  bits
    I_c(R>A)        = -0.806313  bits
    Discord D(R|A)  = 0.006402  bits
    Discord D(A|R)  = 0.000000  bits
Min I(R:B)+I(R:E) = -0.000000000 


(True, -8.881784197001252e-16)

### CQ-2 · Asymmetric weights, orthogonal pure $A$

$$\rho_{CQ2} = \frac{3}{4}|0\rangle\langle 0|_R \otimes |0\rangle\langle 0|_A + \frac{1}{4}|1\rangle\langle 1|_R \otimes |1\rangle\langle 1|_A$$


In [43]:
rho_CQ2 = sep_cq(0.75, ket0, ket0, 0.25, ket1, ket1)
find_perfect_hiding(rho_CQ2, "CQ-2: Asymmetric weights, orthogonal pure A")


State   : CQ-2: Asymmetric weights, orthogonal pure A

  Outputs for CQ-2: Asymmetric weights, orthogonal pure A
    I(R:A)          = 0.811278  bits
    I_c(R>A)        = 0.000000  bits
    Discord D(R|A)  = 0.000000  bits
    Discord D(A|R)  = 0.000000  bits
Min I(R:B)+I(R:E) = -0.000000000 


(True, -3.774758283725532e-15)

### CQ-3 · Equal weights, orthogonal pure $A$ tilted at $\pi/6$

$$|\phi\rangle = \cos\!\frac{\pi}{6}|0\rangle + \sin\!\frac{\pi}{6}|1\rangle, \qquad |\phi^\perp\rangle = -\sin\!\frac{\pi}{6}|0\rangle + \cos\!\frac{\pi}{6}|1\rangle$$

$$\rho_{CQ3} = \frac{1}{2}|0\rangle\langle 0|_R \otimes |\phi\rangle\langle\phi|_A + \frac{1}{2}|1\rangle\langle 1|_R \otimes |\phi^\perp\rangle\langle\phi^\perp|_A$$


In [44]:
phi_cq3   = np.array([np.cos(np.pi/6),  np.sin(np.pi/6)], dtype=complex)
phi_cq3_p = np.array([-np.sin(np.pi/6), np.cos(np.pi/6)], dtype=complex)

rho_CQ3 = sep_cq(0.5, ket0, phi_cq3, 0.5, ket1, phi_cq3_p)
find_perfect_hiding(rho_CQ3, "CQ-3: Orthogonal pure A tilted pi/6")


State   : CQ-3: Orthogonal pure A tilted pi/6

  Outputs for CQ-3: Orthogonal pure A tilted pi/6
    I(R:A)          = 1.000000  bits
    I_c(R>A)        = 0.000000  bits
    Discord D(R|A)  = 0.000000  bits
    Discord D(A|R)  = 0.000000  bits
Min I(R:B)+I(R:E) = -0.000000000 


(True, -8.881784197001252e-16)

### CQ-4 · Equal weights, commuting diagonal mixed $A$

$$\rho_A^{(0)} = \begin{pmatrix}0.8 & 0\\ 0 & 0.2\end{pmatrix}, \qquad \rho_A^{(1)} = \begin{pmatrix}0.3 & 0\\ 0 & 0.7\end{pmatrix}$$

$$\rho_{CQ4} = \frac{1}{2}|0\rangle\langle 0|_R \otimes \rho_A^{(0)} + \frac{1}{2}|1\rangle\langle 1|_R \otimes \rho_A^{(1)}$$

In [45]:
rhoA_cq4_0 = np.array([[0.8, 0.0], [0.0, 0.2]], dtype=complex)
rhoA_cq4_1 = np.array([[0.3, 0.0], [0.0, 0.7]], dtype=complex)

rho_CQ4 = (0.5 * np.kron(ket_to_dm(ket0), rhoA_cq4_0)
         + 0.5 * np.kron(ket_to_dm(ket1), rhoA_cq4_1))
find_perfect_hiding(rho_CQ4, "CQ-4: Commuting diagonal mixed A")


State   : CQ-4: Commuting diagonal mixed A

  Outputs for CQ-4: Commuting diagonal mixed A
    I(R:A)          = 0.191165  bits
    I_c(R>A)        = -0.808835  bits
    Discord D(R|A)  = 0.000000  bits
    Discord D(A|R)  = 0.000000  bits
Min I(R:B)+I(R:E) = 0.000000000 


(True, 0.0)

### CQ-5 · Non-commuting mixed $A$, Bloch vectors at $\pi/6$

$$\rho_A^{(0)} = \frac{1}{2}(I + 0.8\,X), \qquad \rho_A^{(1)} = \frac{1}{2}\!\left(I + 0.8\cos\!\frac{\pi}{6}\,X + 0.8\sin\!\frac{\pi}{6}\,Y\right)$$

$$\rho_{CQ5} = \frac{1}{2}|0\rangle\langle 0|_R \otimes \rho_A^{(0)} + \frac{1}{2}|1\rangle\langle 1|_R \otimes \rho_A^{(1)}$$

In [46]:
theta = np.pi / 6
r     = 0.8
_X = np.array([[0,1],[1,0]],    dtype=complex)
_Y = np.array([[0,-1j],[1j,0]], dtype=complex)

rhoA_cq5_0 = 0.5 * (np.eye(2) + r * _X)
rhoA_cq5_1 = 0.5 * (np.eye(2) + r * (np.cos(theta)*_X + np.sin(theta)*_Y))

rho_CQ5 = (0.5 * np.kron(ket_to_dm(ket0), rhoA_cq5_0)
         + 0.5 * np.kron(ket_to_dm(ket1), rhoA_cq5_1))
find_perfect_hiding(rho_CQ5, "CQ-5: Non-commuting mixed A, angle pi/6")


State   : CQ-5: Non-commuting mixed A, angle pi/6

  Outputs for CQ-5: Non-commuting mixed A, angle pi/6
    I(R:A)          = 0.041772  bits
    I_c(R>A)        = -0.958228  bits
    Discord D(R|A)  = 0.010622  bits
    Discord D(A|R)  = 0.000000  bits
Min I(R:B)+I(R:E) = 0.000000000 


(True, 1.9317880628477724e-14)

### CQ-6 · Non-commuting mixed $A$, Bloch vectors perpendicular

$$\rho_A^{(0)} = \frac{1}{2}(I + 0.8\,X), \qquad \rho_A^{(1)} = \frac{1}{2}(I + 0.8\,Y)$$

$$\rho_{CQ6} = \frac{1}{2}|0\rangle\langle 0|_R \otimes \rho_A^{(0)} + \frac{1}{2}|1\rangle\langle 1|_R \otimes \rho_A^{(1)}$$


In [47]:
r    = 0.8
_X = np.array([[0,1],[1,0]],    dtype=complex)
_Y = np.array([[0,-1j],[1j,0]], dtype=complex)

rhoA_cq6_0 = 0.5 * (np.eye(2) + r * _X)
rhoA_cq6_1 = 0.5 * (np.eye(2) + r * _Y)

rho_CQ6 = (0.5 * np.kron(ket_to_dm(ket0), rhoA_cq6_0)
         + 0.5 * np.kron(ket_to_dm(ket1), rhoA_cq6_1))
find_perfect_hiding(rho_CQ6, "CQ-6: Non-commuting mixed A, perpendicular Bloch")


State   : CQ-6: Non-commuting mixed A, perpendicular Bloch

  Outputs for CQ-6: Non-commuting mixed A, perpendicular Bloch
    I(R:A)          = 0.285947  bits
    I_c(R>A)        = -0.714053  bits
    Discord D(R|A)  = 0.040890  bits
    Discord D(A|R)  = 0.000000  bits
Min I(R:B)+I(R:E) = 0.000000000 


(True, 1.1102230246251565e-15)

### CQ-7 · Non-commuting mixed $A$, antipodal Bloch vectors

$$\rho_A^{(0)} = \frac{1}{2}(I + 0.6\,X), \qquad \rho_A^{(1)} = \frac{1}{2}(I - 0.6\,X)$$

$$\rho_{CQ7} = \frac{1}{2}|0\rangle\langle 0|_R \otimes \rho_A^{(0)} + \frac{1}{2}|1\rangle\langle 1|_R \otimes \rho_A^{(1)}$$

In [48]:
r    = 0.6
_X = np.array([[0,1],[1,0]], dtype=complex)

rhoA_cq7_0 = 0.5 * (np.eye(2) + r * _X)
rhoA_cq7_1 = 0.5 * (np.eye(2) - r * _X)

rho_CQ7 = (0.5 * np.kron(ket_to_dm(ket0), rhoA_cq7_0)
         + 0.5 * np.kron(ket_to_dm(ket1), rhoA_cq7_1))
find_perfect_hiding(rho_CQ7, "CQ-7: Non-commuting mixed A, antipodal X Bloch")


State   : CQ-7: Non-commuting mixed A, antipodal X Bloch

  Outputs for CQ-7: Non-commuting mixed A, antipodal X Bloch
    I(R:A)          = 0.278072  bits
    I_c(R>A)        = -0.721928  bits
    Discord D(R|A)  = 0.000000  bits
    Discord D(A|R)  = 0.000000  bits
Min I(R:B)+I(R:E) = -0.000000000 


(True, -2.220446049250313e-16)

### CQ-8 · Asymmetric weights, non-orthogonal pure $A$ on same great circle

$$|\phi_1\rangle = \cos\!\frac{\pi}{5}|0\rangle + \sin\!\frac{\pi}{5}|1\rangle, \qquad |\phi_2\rangle = \cos\!\frac{2\pi}{5}|0\rangle + \sin\!\frac{2\pi}{5}|1\rangle$$

$$\rho_{CQ8} = \frac{7}{10}|0\rangle\langle 0|_R \otimes |\phi_1\rangle\langle\phi_1|_A + \frac{3}{10}|1\rangle\langle 1|_R \otimes |\phi_2\rangle\langle\phi_2|_A$$


In [49]:
phi_cq8_1 = np.array([np.cos(np.pi/5),   np.sin(np.pi/5)],   dtype=complex)
phi_cq8_2 = np.array([np.cos(2*np.pi/5), np.sin(2*np.pi/5)], dtype=complex)

rho_CQ8 = sep_cq(0.7, ket0, phi_cq8_1, 0.3, ket1, phi_cq8_2)
find_perfect_hiding(rho_CQ8, "CQ-8: Asymmetric weights, non-orthogonal pure A")


State   : CQ-8: Asymmetric weights, non-orthogonal pure A

  Outputs for CQ-8: Asymmetric weights, non-orthogonal pure A
    I(R:A)          = 0.397779  bits
    I_c(R>A)        = -0.483511  bits
    Discord D(R|A)  = 0.161480  bits
    Discord D(A|R)  = 0.000000  bits
Min I(R:B)+I(R:E) = 0.000000000 


(True, 4.440892098500626e-16)

### CQ-9 · One pure $A$, one maximally mixed $A$

$$\rho_A^{(0)} = |{+}\rangle\langle{+}|, \qquad \rho_A^{(1)} = \frac{I}{2}$$

$$\rho_{CQ9} = \frac{1}{2}|0\rangle\langle 0|_R \otimes |{+}\rangle\langle{+}|_A + \frac{1}{2}|1\rangle\langle 1|_R \otimes \frac{I_A}{2}$$

In [50]:
rhoA_cq9_0 = ket_to_dm(ket_p)
rhoA_cq9_1 = np.eye(2, dtype=complex) / 2

rho_CQ9 = (0.5 * np.kron(ket_to_dm(ket0), rhoA_cq9_0)
         + 0.5 * np.kron(ket_to_dm(ket1), rhoA_cq9_1))
find_perfect_hiding(rho_CQ9, "CQ-9: Pure A vs maximally mixed A")


State   : CQ-9: Pure A vs maximally mixed A

  Outputs for CQ-9: Pure A vs maximally mixed A
    I(R:A)          = 0.311278  bits
    I_c(R>A)        = -0.688722  bits
    Discord D(R|A)  = 0.000000  bits
    Discord D(A|R)  = 0.000000  bits
Min I(R:B)+I(R:E) = -0.000000000 


(True, -4.440892098500626e-16)

## CATEGORY 7 : Three-Component CQ States

### 3C-1 · Z₃-Trisymmetric A Frame, Non-Orthogonal R

**Definition:**

$$\rho_{3C1} = \frac{1}{3}\sum_{k=0}^{2}\,\rho_R^{(k)}\otimes|\phi_k\rangle\langle\phi_k|_A$$

where the A-states form a **tight equatorial Z₃ frame**:

$$|\phi_k\rangle = \frac{1}{\sqrt{2}}\!\left(|0\rangle + e^{i2\pi k/3}|1\rangle\right), \quad k=0,1,2$$

and the R-states are non-orthogonal:
$$\rho_R^{(0)}=|0\rangle\langle0|,\quad\rho_R^{(1)}=|1\rangle\langle1|,\quad\rho_R^{(2)}=|{+}\rangle\langle{+}|$$

**Key properties:**
- $\sum_{k=0}^{2}|\phi_k\rangle\langle\phi_k| = \tfrac{3}{2}I$ (tight frame $\Rightarrow$ $\rho_A = \tfrac{I}{2}$, maximally mixed) 
- $S(\rho_A) = 1$ bit, $S(\rho_R) = \log_2 3 - \tfrac{1}{3}$ bits
- A is **overcomplete** in $\mathbb{C}^2$: no projective measurement perfectly distinguishes $\{|\phi_k\rangle\}$ $\Rightarrow$ $D > 0$
- Three components exceed the dimension of A, creating genuine multi-term correlations

In [51]:
phi_3c1 = [np.array([1, np.exp(1j*2*np.pi*k/3)], dtype=complex)/np.sqrt(2) for k in range(3)]
R_3c1   = [dm0, dm1, ket_to_dm(ket_p)]
rho_3C1 = sum((1/3)*np.kron(R_3c1[k], ket_to_dm(phi_3c1[k])) for k in range(3))
find_perfect_hiding(rho_3C1, "3C-1: Z3-trisymmetric A frame, non-orthogonal R")


State   : 3C-1: Z3-trisymmetric A frame, non-orthogonal R

  Outputs for 3C-1: Z3-trisymmetric A frame, non-orthogonal R
    I(R:A)          = 0.459148  bits
    I_c(R>A)        = -0.459148  bits
    Discord D(R|A)  = 0.190875  bits
    Discord D(A|R)  = 0.203155  bits
Min I(R:B)+I(R:E) = 0.000000000 


(True, 1.5543122344752192e-15)

### 3C-2 · Three-Component, Asymmetric Weights, Fully Complex

**Definition:**

$$\rho_{3C2} = 0.5\,|0\rangle\langle0|_R\otimes|\phi_0\rangle\langle\phi_0|_A + 0.3\,|1\rangle\langle1|_R\otimes|\phi_1\rangle\langle\phi_1|_A + 0.2\,|{+_i}\rangle\langle{+_i}|_R\otimes|\phi_2\rangle\langle\phi_2|_A$$

where:

$$|\phi_k\rangle = \cos\!\tfrac{\pi}{5}|0\rangle + e^{i2\pi k/3}\sin\!\tfrac{\pi}{5}|1\rangle, \quad k=0,1,2$$

**Key properties:**
- Asymmetric probability vector $(0.5, 0.3, 0.2)$ breaks all symmetry
- R has both real ($|0\rangle,|1\rangle$) and complex ($|{+_i}\rangle$) components
- A states are non-orthogonal, non-uniform on the Bloch sphere
- Tests the optimizer on a state with no obvious symmetry axis

In [52]:
phi_3c2 = [np.array([np.cos(np.pi/5),
                      np.exp(1j*2*np.pi*k/3)*np.sin(np.pi/5)], dtype=complex)
           for k in range(3)]
R_3c2   = [dm0, dm1, ket_to_dm(ket_pi)]
ps_3c2  = [0.5, 0.3, 0.2]
rho_3C2 = sum(ps_3c2[k]*np.kron(R_3c2[k], ket_to_dm(phi_3c2[k])) for k in range(3))
find_perfect_hiding(rho_3C2, "3C-2: 3-component asymmetric, fully complex")


State   : 3C-2: 3-component asymmetric, fully complex

  Outputs for 3C-2: 3-component asymmetric, fully complex
    I(R:A)          = 0.473232  bits
    I_c(R>A)        = -0.468265  bits
    Discord D(R|A)  = 0.136444  bits
    Discord D(A|R)  = 0.124342  bits
Min I(R:B)+I(R:E) = 0.000000000 


(True, 1.1102230246251565e-15)

### 3C-3 · Three-Component: Two Pure A, One Maximally Mixed A

**Definition:**

C

**Key properties:**
- Third component has $\rho_A^{(2)} = I/2$: perfectly uninformative about A
- Tests hiding when one arm carries no classical information about A
- $I(R:A) > 0$ because the first two terms establish R–A correlations
- The maximally mixed A term introduces genuine rank structure in the global state

In [53]:
rho_3C3 = (0.5 * np.kron(dm0,              ket_to_dm(ket_p))
         + 0.3 * np.kron(dm1,              ket_to_dm(ket_mi))
         + 0.2 * np.kron(ket_to_dm(ket_pi), np.eye(2, dtype=complex)/2))
find_perfect_hiding(rho_3C3, "3C-3: 3-component, one maximally mixed A arm")


State   : 3C-3: 3-component, one maximally mixed A arm

  Outputs for 3C-3: 3-component, one maximally mixed A arm
    I(R:A)          = 0.310442  bits
    I_c(R>A)        = -0.631055  bits
    Discord D(R|A)  = 0.069478  bits
    Discord D(A|R)  = 0.036992  bits
Min I(R:B)+I(R:E) = 0.000000000 


(True, 2.220446049250313e-15)

### 3C-4 · Three-Component SIC-3 A Frame (Partial SIC)

**Definition:**

$$\rho_{3C4} = \frac{1}{3}\sum_{k=0}^{2}\rho_R^{(k)}\otimes|\psi_k\rangle\langle\psi_k|_A$$

where the A-states are the first three states of a **d=2 SIC-POVM**:

$$|\psi_0\rangle = |0\rangle, \quad |\psi_1\rangle = \frac{|0\rangle+\sqrt{2}\,|1\rangle}{\sqrt{3}}, \quad |\psi_2\rangle = \frac{|0\rangle+\sqrt{2}\,e^{i2\pi/3}|1\rangle}{\sqrt{3}}$$

with $|\langle\psi_j|\psi_k\rangle|^2 = \tfrac{1}{3}$ for all $j\neq k$ (equiangular frame).

**Key properties:**
- A frame is maximally spread yet equiangular: hardest possible classical measurement problem
- The three A-states span $\mathbb{C}^2$ with equal pairwise overlaps
- R-states: $|0\rangle, |1\rangle, |{+}\rangle$ — non-orthogonal in R
- Explores the border between achievable and blocked hiding

In [54]:
phi_sic3 = [
    np.array([1, 0],                                              dtype=complex),
    np.array([1, np.sqrt(2)],                                     dtype=complex) / np.sqrt(3),
    np.array([1, np.sqrt(2)*np.exp(1j*2*np.pi/3)],              dtype=complex) / np.sqrt(3),
]
R_3c4 = [dm0, dm1, ket_to_dm(ket_p)]
rho_3C4 = sum((1/3)*np.kron(R_3c4[k], ket_to_dm(phi_sic3[k])) for k in range(3))
find_perfect_hiding(rho_3C4, "3C-4: SIC-3 equiangular A frame")


State   : 3C-4: SIC-3 equiangular A frame

  Outputs for 3C-4: SIC-3 equiangular A frame
    I(R:A)          = 0.422291  bits
    I_c(R>A)        = -0.496005  bits
    Discord D(R|A)  = 0.185945  bits
    Discord D(A|R)  = 0.185945  bits
Min I(R:B)+I(R:E) = 0.000000000 


(True, 1.3322676295501878e-15)

### 3C-5 · Three Non-Commuting Mixed A Components (Bloch Axes)

**Definition:**

$$\rho_{3C5} = \frac{1}{3}|0\rangle\langle0|_R\otimes\rho_A^X + \frac{1}{3}|1\rangle\langle1|_R\otimes\rho_A^Y + \frac{1}{3}|{+}\rangle\langle{+}|_R\otimes\rho_A^{-Z}$$

where each A-component points along a different Bloch axis:

$$\rho_A^X = \tfrac{1}{2}(I+0.7\,X), \quad \rho_A^Y = \tfrac{1}{2}(I+0.7\,Y), \quad \rho_A^{-Z} = \tfrac{1}{2}(I-0.5\,Z)$$

**Key properties:**
- Three A-components are **pairwise non-commuting**: $[\rho_A^X,\rho_A^Y]\neq 0$, etc.
- No single projective measurement simultaneously diagonalises all three $\Rightarrow$ genuine discord
- Bloch radii $r=0.7, 0.7, 0.5$: mixed A with non-trivial purity structure
- Non-orthogonal R (third component $|{+}\rangle$) adds additional complexity

In [55]:
_X = np.array([[0,1],[1,0]],    dtype=complex)
_Y = np.array([[0,-1j],[1j,0]], dtype=complex)
_Z = np.array([[1,0],[0,-1]],   dtype=complex)

rhoA_3c5 = [
    0.5*(np.eye(2) + 0.7*_X),   # Bloch vector along X
    0.5*(np.eye(2) + 0.7*_Y),   # Bloch vector along Y
    0.5*(np.eye(2) - 0.5*_Z),   # Bloch vector along -Z
]
R_3c5 = [dm0, dm1, ket_to_dm(ket_p)]
rho_3C5 = sum((1/3)*np.kron(R_3c5[k], rhoA_3c5[k]) for k in range(3))
find_perfect_hiding(rho_3C5, "3C-5: 3 non-commuting mixed A (Bloch axes X, Y, -Z)")


State   : 3C-5: 3 non-commuting mixed A (Bloch axes X, Y, -Z)

  Outputs for 3C-5: 3 non-commuting mixed A (Bloch axes X, Y, -Z)
    I(R:A)          = 0.113062  bits
    I_c(R>A)        = -0.805234  bits
    Discord D(R|A)  = 0.029668  bits
    Discord D(A|R)  = 0.028821  bits
Min I(R:B)+I(R:E) = 0.000000000 


(True, 3.552713678800501e-15)

### 4C-1 · BB84-Symmetric A States (Four Cardinal Bloch Directions)

**Definition:**

$$\rho_{4C1} = \frac{1}{4}\sum_{k=0}^{3}\rho_R^{(k)}\otimes|\varphi_k\rangle\langle\varphi_k|_A$$

with A-states covering all four BB84 directions (poles + two equatorial):

$$|\varphi_0\rangle=|0\rangle,\quad|\varphi_1\rangle=|1\rangle,\quad|\varphi_2\rangle=|{+}\rangle,\quad|\varphi_3\rangle=|{-}\rangle$$

and R-states: $\rho_R^{(k)} = |0\rangle\langle0|,\,|1\rangle\langle1|,\,|{+}\rangle\langle{+}|,\,|{-}\rangle\langle{-}|$

**Key properties:**
- A-states span all 6 faces of the Bloch octahedron via their 4 directions
- $\sum_k|\varphi_k\rangle\langle\varphi_k| = |0\rangle\langle0|+|1\rangle\langle1|+|{+}\rangle\langle{+}|+|{-}\rangle\langle{-}| = \tfrac{3}{2}I + \tfrac{1}{2}(Z + X) \neq cI$: frame is **not** tight
- $\rho_A = \tfrac{1}{2}I$ despite non-tight frame (by symmetry $Z$ and $X$ contributions cancel in the mixture)
- No single POVM simultaneously distinguishes all four A-states without error

In [56]:
phi_4c1 = [ket0, ket1, ket_p, ket_m]
R_4c1   = [dm0, dm1, ket_to_dm(ket_p), ket_to_dm(ket_m)]
rho_4C1 = sum(0.25*np.kron(R_4c1[k], ket_to_dm(phi_4c1[k])) for k in range(4))
find_perfect_hiding(rho_4C1, "4C-1: BB84-symmetric A states (4 cardinal Bloch)")


State   : 4C-1: BB84-symmetric A states (4 cardinal Bloch)

  Outputs for 4C-1: BB84-symmetric A states (4 cardinal Bloch)
    I(R:A)          = 0.500000  bits
    I_c(R>A)        = -0.500000  bits
    Discord D(R|A)  = 0.311278  bits
    Discord D(A|R)  = 0.311278  bits
Min I(R:B)+I(R:E) = -0.000000000 


(True, -4.440892098500626e-16)

### 4C-2 · Z₄-Equatorial Tight Frame (Four Equatorial A States)

**Definition:**

$$\rho_{4C2} = \frac{1}{4}\sum_{k=0}^{3}\rho_R^{(k)}\otimes|\varphi_k\rangle\langle\varphi_k|_A$$

where the A-states form a **Z₄-symmetric equatorial tight frame**:

$$|\varphi_k\rangle = \frac{|0\rangle + i^k|1\rangle}{\sqrt{2}},\quad k=0,1,2,3 \quad\Longrightarrow\quad |{+}\rangle,\,|{+_i}\rangle,\,|{-}\rangle,\,|{-_i}\rangle$$

and R-states: $\rho_R^{(k)} = |0\rangle\langle0|,\,|1\rangle\langle1|,\,|{+}\rangle\langle{+}|,\,|{+_i}\rangle\langle{+_i}|$

**Key properties:**
- $\sum_{k=0}^{3}|\varphi_k\rangle\langle\varphi_k| = 2I$ (tight frame $\Rightarrow$ $\rho_A = I/2$ exactly)
- A-states lie **entirely on the equator** of the Bloch sphere: no Z-axis information
- Four equatorial states cannot be perfectly distinguished by any projective measurement
- Coherent information $I_c(R\triangleright A) = S(\rho_A)-S(\rho_{RA}) = 1 - S(\rho_{RA})$

In [57]:
phi_4c2 = [np.array([1, 1j**k], dtype=complex)/np.sqrt(2) for k in range(4)]
R_4c2   = [dm0, dm1, ket_to_dm(ket_p), ket_to_dm(ket_pi)]
rho_4C2 = sum(0.25*np.kron(R_4c2[k], ket_to_dm(phi_4c2[k])) for k in range(4))
find_perfect_hiding(rho_4C2, "4C-2: Z4-equatorial tight frame A")


State   : 4C-2: Z4-equatorial tight frame A

  Outputs for 4C-2: Z4-equatorial tight frame A
    I(R:A)          = 0.215594  bits
    I_c(R>A)        = -0.692258  bits
    Discord D(R|A)  = 0.068806  bits
    Discord D(A|R)  = 0.075762  bits
Min I(R:B)+I(R:E) = -0.000000000 


(True, -1.1102230246251565e-15)

### 4C-3 · SIC-POVM A States (Tetrahedral Frame, d=2)

**Definition:**

$$\rho_{4C3} = \frac{1}{4}\sum_{k=0}^{3}\rho_R^{(k)}\otimes|\psi_k\rangle\langle\psi_k|_A$$

The four A-states are the **symmetric informationally complete (SIC)** states in $\mathbb{C}^2$:

$$|\psi_0\rangle = |0\rangle, \quad |\psi_k\rangle = \frac{|0\rangle + \sqrt{2}\,e^{i2\pi(k-1)/3}|1\rangle}{\sqrt{3}},\quad k=1,2,3$$

with the defining SIC property: $|\langle\psi_j|\psi_k\rangle|^2 = \tfrac{1}{d+1} = \tfrac{1}{3}$ for all $j\neq k$.

R-states: $|0\rangle\langle0|,\,|1\rangle\langle1|,\,|{+}\rangle\langle{+}|,\,|{+_i}\rangle\langle{+_i}|$

**Key properties:**
- $\sum_{k=0}^{3}|\psi_k\rangle\langle\psi_k| = 2I$: SIC states form a **tight frame** $\Rightarrow$ $\rho_A = I/2$
- SIC states are the **most informative** overcomplete measurements in $\mathbb{C}^2$
- The four Bloch vectors point to vertices of a regular tetrahedron inscribed in the Bloch sphere
- This is the quantum analogue of a regular simplex — optimal tomographic design

In [58]:
omega_sic = np.exp(1j * 2*np.pi/3)
phi_sic4  = [
    np.array([1, 0],                              dtype=complex),
    np.array([1, np.sqrt(2)],                     dtype=complex) / np.sqrt(3),
    np.array([1, np.sqrt(2)*omega_sic],           dtype=complex) / np.sqrt(3),
    np.array([1, np.sqrt(2)*omega_sic**2],        dtype=complex) / np.sqrt(3),
]
R_4c3 = [dm0, dm1, ket_to_dm(ket_p), ket_to_dm(ket_pi)]
rho_4C3 = sum(0.25*np.kron(R_4c3[k], ket_to_dm(phi_sic4[k])) for k in range(4))
find_perfect_hiding(rho_4C3, "4C-3: SIC-POVM tetrahedral A frame")


State   : 4C-3: SIC-POVM tetrahedral A frame

  Outputs for 4C-3: SIC-POVM tetrahedral A frame
    I(R:A)          = 0.348867  bits
    I_c(R>A)        = -0.558985  bits
    Discord D(R|A)  = 0.218972  bits
    Discord D(A|R)  = 0.225058  bits
Min I(R:B)+I(R:E) = 0.069589909 


(False, 0.06958990887597594)

### 4C-4 · Four-Component Asymmetric, Complex R and A

**Definition:**

$$\rho_{4C4} = 0.4\,|{+_i}\rangle\langle{+_i}|_R\otimes|\phi_0\rangle\langle\phi_0|_A + 0.3\,|{-_i}\rangle\langle{-_i}|_R\otimes|\phi_1\rangle\langle\phi_1|_A + 0.2\,|\xi\rangle\langle\xi|_R\otimes|\phi_2\rangle\langle\phi_2|_A + 0.1\,|0\rangle\langle0|_R\otimes|\phi_3\rangle\langle\phi_3|_A$$

where:

$$|\phi_0\rangle = |{+}\rangle,\quad |\phi_1\rangle = |{+_i}\rangle,\quad |\phi_2\rangle = \cos\tfrac{\pi}{5}|0\rangle + e^{i\pi/4}\sin\tfrac{\pi}{5}|1\rangle, \quad |\phi_3\rangle = \cos\tfrac{\pi}{7}|0\rangle + e^{-i\pi/3}\sin\tfrac{\pi}{7}|1\rangle$$

$$|\xi\rangle = \frac{|0\rangle + e^{i\pi/5}|1\rangle}{\sqrt{2}}$$

**Key properties:**
- Completely broken symmetry: no two weights equal, no two A-states related by obvious symmetry
- Both R and A subsystems carry complex phases on every component
- Tests the optimizer under maximally generic asymmetric conditions with 4 terms

In [59]:
phi_4c4 = [
    ket_p,
    ket_pi,
    np.array([np.cos(np.pi/5), np.exp(1j*np.pi/4)*np.sin(np.pi/5)], dtype=complex),
    np.array([np.cos(np.pi/7), np.exp(-1j*np.pi/3)*np.sin(np.pi/7)], dtype=complex),
]
xi_4c4  = np.array([1, np.exp(1j*np.pi/5)], dtype=complex) / np.sqrt(2)
R_4c4   = [ket_to_dm(ket_pi), ket_to_dm(ket_mi), ket_to_dm(xi_4c4), dm0]
ps_4c4  = [0.4, 0.3, 0.2, 0.1]
rho_4C4 = sum(ps_4c4[k]*np.kron(R_4c4[k], ket_to_dm(phi_4c4[k])) for k in range(4))
find_perfect_hiding(rho_4C4, "4C-4: 4-component asymmetric, complex R and A")


State   : 4C-4: 4-component asymmetric, complex R and A

  Outputs for 4C-4: 4-component asymmetric, complex R and A
    I(R:A)          = 0.243995  bits
    I_c(R>A)        = -0.694897  bits
    Discord D(R|A)  = 0.061121  bits
    Discord D(A|R)  = 0.019097  bits
Min I(R:B)+I(R:E) = 0.003759641 


(False, 0.003759641131131808)

### 5C-1 · Z₅-Pentagonal Equatorial Frame

**Definition:**

$$\rho_{5C1} = \frac{1}{5}\sum_{k=0}^{4}\rho_R^{(k)}\otimes|\phi_k\rangle\langle\phi_k|_A$$

where the A-states form a **Z₅-symmetric equatorial frame**:

$$|\phi_k\rangle = \frac{|0\rangle + e^{i2\pi k/5}|1\rangle}{\sqrt{2}}, \quad k=0,1,2,3,4$$

and R-states: $|0\rangle\langle0|,\,|1\rangle\langle1|,\,|{+}\rangle\langle{+}|,\,|{+_i}\rangle\langle{+_i}|,\,|{-}\rangle\langle{-}|$

**Key properties:**
- $\sum_{k=0}^{4}|\phi_k\rangle\langle\phi_k| = \tfrac{5}{2}I$ (tight Z₅ frame $\Rightarrow$ $\rho_A = I/2$)
- Five A-states exceed the dimension of A by a factor of 2.5: maximally overcomplete
- Pentagon-symmetric: 5-fold rotational symmetry on the Bloch equator
- Any measurement on A disturbs the state: quantum discord is necessarily positive
- The most components studied: stress-tests the optimizer landscape

In [60]:
phi_5c1 = [np.array([1, np.exp(1j*2*np.pi*k/5)], dtype=complex)/np.sqrt(2) for k in range(5)]
R_5c1   = [dm0, dm1, ket_to_dm(ket_p), ket_to_dm(ket_pi), ket_to_dm(ket_m)]
rho_5C1 = sum(0.2*np.kron(R_5c1[k], ket_to_dm(phi_5c1[k])) for k in range(5))
find_perfect_hiding(rho_5C1, "5C-1: Z5-pentagonal equatorial frame")


State   : 5C-1: Z5-pentagonal equatorial frame

  Outputs for 5C-1: Z5-pentagonal equatorial frame
    I(R:A)          = 0.190533  bits
    I_c(R>A)        = -0.780418  bits
    Discord D(R|A)  = 0.038768  bits
    Discord D(A|R)  = 0.041022  bits
Min I(R:B)+I(R:E) = 0.000000000 


(True, 1.7763568394002505e-15)

### 5C-2 · Five-Component Asymmetric, Fully Complex

**Definition:**

$$\rho_{5C2} = \sum_{k=1}^{5} p_k\,\rho_R^{(k)}\otimes|\phi_k\rangle\langle\phi_k|_A$$

with weights $\mathbf{p} = (0.30, 0.25, 0.20, 0.15, 0.10)$ and:

$$|\phi_k\rangle = \cos\!\tfrac{k\pi}{8}|0\rangle + e^{ik\pi/4}\sin\!\tfrac{k\pi}{8}|1\rangle, \quad k=1,\ldots,5$$

R-states: $|0\rangle\langle0|,\,|{+_i}\rangle\langle{+_i}|,\,|1\rangle\langle1|,\,|{-_i}\rangle\langle{-_i}|,\,|{+}\rangle\langle{+}|$

**Key properties:**
- Strictly monotone weights: breaks all symmetry between components
- A-states trace a helical path on the Bloch sphere (polar angle $k\pi/8$, azimuth $k\pi/4$)
- Alternating real/complex R-states: $|0\rangle, |{+_i}\rangle, |1\rangle, |{-_i}\rangle, |{+}\rangle$
- No closed-form $\rho_A$ — must be computed numerically
- Most asymmetric 5-component test case

In [62]:
ps_5c2  = [0.30, 0.25, 0.20, 0.15, 0.10]
phi_5c2 = [np.array([np.cos((k+1)*np.pi/8),
                      np.exp(1j*(k+1)*np.pi/4)*np.sin((k+1)*np.pi/8)], dtype=complex)
           for k in range(5)]
R_5c2   = [dm0, ket_to_dm(ket_pi), dm1, ket_to_dm(ket_mi), ket_to_dm(ket_p)]
rho_5C2 = sum(ps_5c2[k]*np.kron(R_5c2[k], ket_to_dm(phi_5c2[k])) for k in range(5))
find_perfect_hiding(rho_5C2, "5C-2: 5-component asymmetric, fully complex")


State   : 5C-2: 5-component asymmetric, fully complex

  Outputs for 5C-2: 5-component asymmetric, fully complex
    I(R:A)          = 0.302216  bits
    I_c(R>A)        = -0.676034  bits
    Discord D(R|A)  = 0.135485  bits
    Discord D(A|R)  = 0.109251  bits
Min I(R:B)+I(R:E) = 0.005145637 


(False, 0.00514563722625172)

### 5C-3 · Five Mixed A Components Along Bloch Cardinal Axes

**Definition:**

$$\rho_{5C3} = \frac{1}{5}\sum_{k=0}^{4}\rho_R^{(k)}\otimes\rho_A^{(k)}$$

Each A-component points along a **different cardinal Bloch axis** (radius $r=0.6$):

$$\rho_A^{(k)} = \frac{1}{2}\!\left(I + 0.6\,\hat{n}_k\cdot\boldsymbol{\sigma}\right), \quad \hat{n}_k \in \{+X,\,+Y,\,-X,\,-Y,\,+Z\}$$

R-states: $|0\rangle\langle0|,\,|1\rangle\langle1|,\,|{+}\rangle\langle{+}|,\,|{+_i}\rangle\langle{+_i}|,\,|{-}\rangle\langle{-}|$

**Key properties:**
- Five directions cover all four equatorial cardinal directions plus the north pole
- A-components are **pairwise non-commuting** (each along a different axis)
- Mixed A (not pure): Bloch radius $r=0.6 < 1$, so each $\rho_A^{(k)}$ is genuinely mixed
- $\rho_A = \tfrac{1}{5}\sum_k\rho_A^{(k)} \approx \tfrac{I}{2}$ (Z-axis breaks exact equality)
- Highest complexity 5-component test: 5 distinct mixed non-commuting A-components

In [63]:
_X = np.array([[0,1],[1,0]],    dtype=complex)
_Y = np.array([[0,-1j],[1j,0]], dtype=complex)
_Z = np.array([[1,0],[0,-1]],   dtype=complex)

rhoA_5c3 = [
    0.5*(np.eye(2) + 0.6*_X),    # +X axis
    0.5*(np.eye(2) + 0.6*_Y),    # +Y axis
    0.5*(np.eye(2) - 0.6*_X),    # -X axis
    0.5*(np.eye(2) - 0.6*_Y),    # -Y axis
    0.5*(np.eye(2) + 0.6*_Z),    # +Z axis (north pole)
]
R_5c3 = [dm0, dm1, ket_to_dm(ket_p), ket_to_dm(ket_pi), ket_to_dm(ket_m)]
rho_5C3 = sum(0.2*np.kron(R_5c3[k], rhoA_5c3[k]) for k in range(5))
find_perfect_hiding(rho_5C3, "5C-3: 5 mixed A on Bloch cardinal axes (+X,+Y,-X,-Y,+Z)")


State   : 5C-3: 5 mixed A on Bloch cardinal axes (+X,+Y,-X,-Y,+Z)

  Outputs for 5C-3: 5 mixed A on Bloch cardinal axes (+X,+Y,-X,-Y,+Z)
    I(R:A)          = 0.052751  bits
    I_c(R>A)        = -0.918200  bits
    Discord D(R|A)  = 0.018919  bits
    Discord D(A|R)  = 0.019190  bits
Min I(R:B)+I(R:E) = 0.002478522 


(False, 0.0024785224526731042)

## CATEGORY 10 :  Pure Entangled States 2

### PE-6 · Full-Support Entangled State (All Four Amplitudes Nonzero)

**Definition:**

$$|\psi_{PE6}\rangle = \frac{1}{2}\!\left(|00\rangle + |01\rangle + |10\rangle - |11\rangle\right)$$

**Entanglement test:** $ad - bc = \tfrac{1}{2}\cdot(-\tfrac{1}{2}) - \tfrac{1}{2}\cdot\tfrac{1}{2} = -\tfrac{1}{2} \neq 0$ $\Rightarrow$ entangled.

**Schmidt decomposition:** $\rho_R = \mathrm{Tr}_A(|\psi\rangle\langle\psi|) = I/2$ $\Rightarrow$ **maximally entangled** with Schmidt coefficients $(\tfrac{1}{\sqrt{2}}, \tfrac{1}{\sqrt{2}})$.

**Key difference from Bell states:** All four computational basis states have nonzero amplitude. This state is locally equivalent to $|\Phi^+\rangle$ via a Hadamard on A, but its matrix representation is **dense** — no zeros in the ket — making it a harder numerical target.

In [64]:
psi_PE6 = np.array([1, 1, 1, -1], dtype=complex) / 2.0
rho_PE6 = ket_to_dm(psi_PE6)
find_perfect_hiding(rho_PE6, "PE-6: Full-support entangled (00+01+10-11)/2")


State   : PE-6: Full-support entangled (00+01+10-11)/2

  Outputs for PE-6: Full-support entangled (00+01+10-11)/2
    I(R:A)          = 2.000000  bits
    I_c(R>A)        = 1.000000  bits
    Discord D(R|A)  = 1.000000  bits
    Discord D(A|R)  = 1.000000  bits
Min I(R:B)+I(R:E) = 2.000000000 


(False, 1.9999999999999984)

### PE-7 · Highly Asymmetric Schmidt Coefficients $(\varepsilon = 0.05)$

**Definition:**

$$|\psi_{PE7}\rangle = \sqrt{0.05}\,|00\rangle + \sqrt{0.95}\,e^{i\pi/7}|11\rangle$$

**Density matrix:**

$$\rho_{PE7} = \begin{pmatrix}0.05&0&0&\sqrt{0.0475}\,e^{-i\pi/7}\\0&0&0&0\\0&0&0&0\\\sqrt{0.0475}\,e^{i\pi/7}&0&0&0.95\end{pmatrix}$$

**Key properties:**
- Schmidt coefficients $\lambda_1 = \sqrt{0.05}$, $\lambda_2 = \sqrt{0.95}$: entanglement concentrated on one term
- $S(\rho_R) = -0.05\log_2(0.05) - 0.95\log_2(0.95) \approx 0.286$ bits: low entanglement
- Irrational phase $e^{i\pi/7}$ avoids any accidental symmetry with the existing PE-5 ($e^{i\pi/4}$)
- Tests whether the optimizer can find the hiding limit at very low entanglement, where the cost landscape may be flat

In [65]:
psi_PE7 = np.array([np.sqrt(0.05), 0, 0,
                     np.sqrt(0.95)*np.exp(1j*np.pi/7)], dtype=complex)
rho_PE7 = ket_to_dm(psi_PE7)
find_perfect_hiding(rho_PE7, "PE-7: Asymmetric Schmidt (0.05, 0.95), phase e^{ipi/7}")


State   : PE-7: Asymmetric Schmidt (0.05, 0.95), phase e^{ipi/7}

  Outputs for PE-7: Asymmetric Schmidt (0.05, 0.95), phase e^{ipi/7}
    I(R:A)          = 0.572794  bits
    I_c(R>A)        = 0.286397  bits
    Discord D(R|A)  = 0.286397  bits
    Discord D(A|R)  = 0.286397  bits
Min I(R:B)+I(R:E) = 0.572793914 


(False, 0.5727939142319096)

### PE-8 · Complex Full-Support Maximally Entangled

**Definition:**

$$|\psi_{PE8}\rangle = \frac{1}{2}\!\left(|00\rangle + i|01\rangle + e^{i\pi/4}|10\rangle - ie^{i\pi/4}|11\rangle\right)$$

**Verification:** $ad - bc = \tfrac{1}{2}\cdot(-\tfrac{i}{2}e^{i\pi/4}) - \tfrac{i}{2}\cdot\tfrac{e^{i\pi/4}}{2} = -\tfrac{i}{2}e^{i\pi/4} \neq 0$ $\Rightarrow$ entangled.

$\rho_R = I/2$: maximally entangled.

**Key properties:**
- Every amplitude carries a distinct complex phase
- Locally equivalent to $|\Phi^+\rangle$ via $(I \otimes U_A)$ where $U_A$ is a single-qubit unitary
- Dense complex matrix: hardest representation for a maximally entangled state
- The phase pattern $e^{i\pi/4}$ on the lower two amplitudes is incommensurate with the upper ones

In [66]:
psi_PE8 = np.array([1, 1j,
                     np.exp(1j*np.pi/4), -1j*np.exp(1j*np.pi/4)], dtype=complex) / 2.0
rho_PE8 = ket_to_dm(psi_PE8)
find_perfect_hiding(rho_PE8, "PE-8: Complex full-support maximally entangled")


State   : PE-8: Complex full-support maximally entangled

  Outputs for PE-8: Complex full-support maximally entangled
    I(R:A)          = 2.000000  bits
    I_c(R>A)        = 1.000000  bits
    Discord D(R|A)  = 1.000000  bits
    Discord D(A|R)  = 1.000000  bits
Min I(R:B)+I(R:E) = 2.000000000 


(False, 1.9999999999999987)

## CATEGORY 11 : New Mixed Entangled States

### NM-6 · All Four Bell States, Unequal Weights

**Definition:**

$$\rho_{NM6} = 0.4\,|\Phi^+\rangle\langle\Phi^+| + 0.3\,|\Psi^-\rangle\langle\Psi^-| + 0.2\,|\Phi^-\rangle\langle\Phi^-| + 0.1\,|\Psi^+\rangle\langle\Psi^+|$$

where $|\Phi^\pm\rangle = (|00\rangle\pm|11\rangle)/\sqrt{2}$, $|\Psi^\pm\rangle = (|01\rangle\pm|10\rangle)/\sqrt{2}$.

**Explicit matrix** (Bell basis diagonal, unequal weights):

$$\rho_{NM6} = \frac{1}{2}\begin{pmatrix}0.6&0&0&0.2\\0&0.4&-0.2&0\\0&-0.2&0.4&0\\0.2&0&0&0.6\end{pmatrix}$$

**Key properties:**
- Mixture of the **entire Bell basis**: no Bell-state coherences survive (off-diagonal Bell blocks vanish)
- Dominant weight on $|\Phi^+\rangle$, decreasing to $|\Psi^+\rangle$: breaks the $U\otimes U$ symmetry of the Werner state
- All four components are maximally entangled yet the mixture is less entangled than any single component
- $\rho_R = \rho_A = I/2$: both marginals maximally mixed

In [67]:
phi_minus    = np.array([1, 0, 0, -1], dtype=complex) / np.sqrt(2)
psi_plus_bell = np.array([0, 1, 1,  0], dtype=complex) / np.sqrt(2)

rho_NM6 = (0.4 * dm_phi_plus
         + 0.3 * dm_psi_minus
         + 0.2 * ket_to_dm(phi_minus)
         + 0.1 * ket_to_dm(psi_plus_bell))
find_perfect_hiding(rho_NM6, "NM-6: All-four-Bell mixture, unequal weights (0.4,0.3,0.2,0.1)")


State   : NM-6: All-four-Bell mixture, unequal weights (0.4,0.3,0.2,0.1)

  Outputs for NM-6: All-four-Bell mixture, unequal weights (0.4,0.3,0.2,0.1)
    I(R:A)          = 0.153561  bits
    I_c(R>A)        = -0.846439  bits
    Discord D(R|A)  = 0.034852  bits
    Discord D(A|R)  = 0.034852  bits
Min I(R:B)+I(R:E) = -0.000000000 


(True, -8.881784197001252e-16)

### NM-8 · Noisy Full-Support Entangled State (Depolarized PE-6)

**Definition:**

$$\rho_{NM8} = 0.7\,|\psi_{PE6}\rangle\langle\psi_{PE6}| + 0.3\,\frac{I_4}{4}$$

where $|\psi_{PE6}\rangle = \tfrac{1}{2}(|00\rangle+|01\rangle+|10\rangle-|11\rangle)$ (maximally entangled).

**Explicit matrix:**

$$\rho_{NM8} = \frac{1}{4}\begin{pmatrix}0.775&0.35&0.35&-0.35\\0.35&0.775&0.35&-0.35\\0.35&0.35&0.775&-0.35\\-0.35&-0.35&-0.35&0.775\end{pmatrix}$$

**Key properties:**
- Depolarization fraction $30\%$: entangled ($0.7 > \tfrac{1}{3}$) but with reduced coherence
- The pure component $|\psi_{PE6}\rangle$ has all four amplitudes nonzero: the noise term fills in the matrix uniformly
- Dense matrix with no zero entries: tests the cost function's ability to handle a fully generic $4\times 4$ input
- Pairs with PE-6 to compare pure vs noisy hiding difficulty

In [68]:
psi_PE6_v = np.array([1, 1, 1, -1], dtype=complex) / 2.0
rho_NM8   = 0.7 * ket_to_dm(psi_PE6_v) + 0.3 * np.eye(4, dtype=complex) / 4
find_perfect_hiding(rho_NM8, "NM-8: Noisy full-support entangled (depolarized PE-6, p=0.7)")


State   : NM-8: Noisy full-support entangled (depolarized PE-6, p=0.7)

  Outputs for NM-8: Noisy full-support entangled (depolarized PE-6, p=0.7)
    I(R:A)          = 0.874191  bits
    I_c(R>A)        = -0.125809  bits
    Discord D(R|A)  = 0.484031  bits
    Discord D(A|R)  = 0.484031  bits
Min I(R:B)+I(R:E) = 0.780319391 


(False, 0.7803193905671999)